# AGN Stochastic Variability Modeling — Inline Notebook

This notebook is now a module-backed AGN stochastic variability workflow. Shared catalog normalization, external context, scoring, and review-export behavior should live in `malca.nuclear`; notebook cells remain for orchestration, diagnostics, and inspection.

## Reusable Nuclear Context Layer

This notebook is now backed by the reusable `malca.nuclear` context and scoring API. Keep exploratory plots and catalog checks here, but put shared AGN/TDE/CLAGN enrichment, scoring, and review-export behavior in modules.


In [ ]:
from malca.nuclear import NuclearContextConfig, normalize_nuclear_targets, run_nuclear_context, score_nuclear_candidates
from malca.nuclear.features import compute_lightcurve_feature_table, compute_nuclear_lightcurve_features

# Example:
# config = NuclearContextConfig(run_dir=RUN_DIR)
# nuclear_context = run_nuclear_context(targets, config)


## Setup

Set `CATALOG_PATHS` to your local AGN/MilliQuas-style catalog(s) when you want to run on real data. The synthetic demo near the bottom runs without network access.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

root_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        p for p in root_candidates
        if (p / 'pyproject.toml').exists()
        and (p / 'malca').is_dir()
    ),
    Path('/Users/calder/code/malca'),
)

for path in (REPO_ROOT, REPO_ROOT / 'malca'):
    text = str(path.resolve())
    if text not in sys.path:
        sys.path.insert(0, text)

RUN_DIR = REPO_ROOT / 'output' / 'runs' / 'agn_stochastic'
CATALOG_PATHS: list[Path] = []  # Fill with one or more local AGN catalogs, e.g. [REPO_ROOT / 'input' / 'milliquas.csv']
RUN_DIR


PosixPath('/Users/calder/code/malca/output/runs/agn_stochastic')

## Notebook Stages

These are the function-call equivalents of the staged workflow.

In [2]:
# Notebook equivalents for the staged workflow below:
print('build_targets_stage(CATALOG_PATHS, run_dir=RUN_DIR)')
print('fetch_lightcurves_stage(run_dir=RUN_DIR, cache_dir=RUN_DIR / \'lightcurves\')')
print('compute_features_stage(run_dir=RUN_DIR, primary_band=\'g\', include_drw=True)')
print('rank_outliers_stage(run_dir=RUN_DIR)')
print('export_review_stage(run_dir=RUN_DIR, min_score=2.0)')


build_targets_stage(CATALOG_PATHS, run_dir=RUN_DIR)
fetch_lightcurves_stage(run_dir=RUN_DIR, cache_dir=RUN_DIR / 'lightcurves')
compute_features_stage(run_dir=RUN_DIR, primary_band='g', include_drw=True)
rank_outliers_stage(run_dir=RUN_DIR)
export_review_stage(run_dir=RUN_DIR, min_score=2.0)


In [3]:
# Available notebook-only stage functions.
stage_functions = [
    'build_targets_stage',
    'fetch_lightcurves_stage',
    'compute_features_stage',
    'rank_outliers_stage',
    'export_review_stage',
    'run_all_stage',
]
for name in stage_functions:
    print(name)


build_targets_stage
fetch_lightcurves_stage
compute_features_stage
rank_outliers_stage
export_review_stage
run_all_stage


## Notebook-Only Source: package marker code

This cell contains the workflow source directly. It is no longer backed by a standalone module.

In [4]:
"""AGN/nuclear variability tooling for MALCA."""

from __future__ import annotations

__all__ = [
    "catalogs",
    "features",
    "photometry",
    "pipeline",
    "stochastic",
    "tde_catalog",
]


## Notebook-Only Source: catalog ingestion

This cell contains the workflow source directly. It is no longer backed by a standalone module.

In [5]:
"""AGN parent-sample ingestion for nuclear variability searches."""

from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd

from malca.io.table_io import read_parquet_table, write_parquet_table


DEFAULT_SOURCE_CATALOG = "agn"

CANONICAL_COLUMNS = [
    "nuclear_target_id",
    "source_catalog",
    "source_name",
    "ra",
    "dec",
    "agn_type",
    "redshift",
    "asassn_id",
    "gaia_id",
    "g_mag",
    "v_mag",
    "r_mag",
    "i_mag",
    "wise_w1",
    "wise_w2",
    "black_hole_mass",
    "bol_luminosity",
    "eddington_ratio",
    "lc_path",
]

COLUMN_ALIASES: dict[str, tuple[str, ...]] = {
    "source_name": (
        "source_name",
        "name",
        "Name",
        "objname",
        "object_name",
        "designation",
        "ID",
        "id",
    ),
    "ra": (
        "ra",
        "RA",
        "ra_deg",
        "RAJ2000",
        "raj2000",
        "RAdeg",
        "raJ2000",
        "R.A.",
        "_RAJ2000",
    ),
    "dec": (
        "dec",
        "DEC",
        "dec_deg",
        "DEJ2000",
        "dej2000",
        "DEdeg",
        "decJ2000",
        "Decl.",
        "_DEJ2000",
    ),
    "agn_type": (
        "agn_type",
        "type",
        "Type",
        "class",
        "Class",
        "otype",
        "simbad_otype",
        "milliquas_type",
    ),
    "redshift": (
        "redshift",
        "z",
        "Z",
        "milliquas_z",
        "spec_z",
        "photo_z",
    ),
    "asassn_id": (
        "asassn_id",
        "asas_sn_id",
        "ASAS_SN_ID",
        "asassn_source_id",
        "source_id_asassn",
    ),
    "gaia_id": (
        "gaia_id",
        "source_id",
        "gaia_source_id",
        "GaiaDR3",
        "Gaia",
    ),
    "g_mag": (
        "g_mag",
        "gmag",
        "g",
        "Pstarss gmag",
        "phot_g_mean_mag",
        "baseline_mag",
    ),
    "v_mag": (
        "v_mag",
        "vmag",
        "Vmag",
        "V",
        "mean_vmag",
    ),
    "r_mag": ("r_mag", "rmag", "r", "Rmag"),
    "i_mag": ("i_mag", "imag", "i", "Imag"),
    "wise_w1": ("wise_w1", "w1", "unwise_w1", "allwise_w1", "W1"),
    "wise_w2": ("wise_w2", "w2", "unwise_w2", "allwise_w2", "W2"),
    "black_hole_mass": (
        "black_hole_mass",
        "mbh",
        "log_mbh",
        "logMBH",
        "M_BH",
    ),
    "bol_luminosity": (
        "bol_luminosity",
        "lbol",
        "log_lbol",
        "Lbol",
    ),
    "eddington_ratio": (
        "eddington_ratio",
        "edd_ratio",
        "log_edd_ratio",
        "lambda_edd",
    ),
    "lc_path": (
        "lc_path",
        "lightcurve_path",
        "asassn_lc_path",
    ),
}

NUMERIC_COLUMNS = {
    "ra",
    "dec",
    "redshift",
    "g_mag",
    "v_mag",
    "r_mag",
    "i_mag",
    "wise_w1",
    "wise_w2",
    "black_hole_mass",
    "bol_luminosity",
    "eddington_ratio",
}

TEXT_COLUMNS = {
    "source_catalog",
    "source_name",
    "agn_type",
    "asassn_id",
    "gaia_id",
    "nuclear_target_id",
    "lc_path",
}


def _pick_column(df: pd.DataFrame, aliases: tuple[str, ...]) -> str | None:
    for alias in aliases:
        if alias in df.columns:
            return alias

    lower_map = {str(col).strip().lower(): col for col in df.columns}
    for alias in aliases:
        key = str(alias).strip().lower()
        if key in lower_map:
            return lower_map[key]
    return None


def _stable_coord_id(source_catalog: str, ra: float, dec: float) -> str:
    ra_txt = f"{float(ra):010.6f}".replace(".", "p").replace("-", "m")
    dec_txt = f"{float(dec):+010.6f}".replace("+", "p").replace(".", "p").replace("-", "m")
    prefix = re.sub(r"[^A-Za-z0-9]+", "_", str(source_catalog).strip().lower()).strip("_") or "agn"
    return f"{prefix}_{ra_txt}_{dec_txt}"


def _clean_text(series: pd.Series) -> pd.Series:
    return series.astype("string").fillna("").str.strip().astype(object)


def normalize_agn_catalog(
    df: pd.DataFrame,
    *,
    source_catalog: str = DEFAULT_SOURCE_CATALOG,
    target_id_prefix: str | None = None,
    drop_duplicate_coords_arcsec: float = 1.0,
) -> pd.DataFrame:
    """Normalize an AGN catalog to MALCA's nuclear target schema.

    The input can be MilliQuas-like, SIMBAD-like, or an existing MALCA table
    with AGN crossmatch columns. Coordinates are required; all other fields are
    optional and are filled with empty strings or NaN.
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)

    out = pd.DataFrame(index=df.index)
    for canonical, aliases in COLUMN_ALIASES.items():
        source_col = _pick_column(df, aliases)
        if source_col is not None:
            out[canonical] = df[source_col]

    for col in CANONICAL_COLUMNS:
        if col in ("nuclear_target_id", "source_catalog"):
            continue
        if col not in out.columns:
            out[col] = np.nan if col in NUMERIC_COLUMNS else ""

    out["source_catalog"] = str(source_catalog)

    for col in NUMERIC_COLUMNS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    for col in TEXT_COLUMNS - {"nuclear_target_id"}:
        if col in out.columns:
            out[col] = _clean_text(out[col])

    valid = (
        np.isfinite(out["ra"].to_numpy(dtype=float, na_value=np.nan))
        & np.isfinite(out["dec"].to_numpy(dtype=float, na_value=np.nan))
        & (out["ra"].astype(float) >= 0.0)
        & (out["ra"].astype(float) < 360.0)
        & (out["dec"].astype(float) >= -90.0)
        & (out["dec"].astype(float) <= 90.0)
    )
    out = out.loc[valid].copy()
    if out.empty:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)

    prefix = target_id_prefix or source_catalog
    names = _clean_text(out["source_name"])
    generated = [
        _stable_coord_id(prefix, ra, dec)
        for ra, dec in zip(out["ra"].to_numpy(dtype=float), out["dec"].to_numpy(dtype=float))
    ]
    safe_names = names.str.replace(r"[^A-Za-z0-9_.:-]+", "_", regex=True).str.strip("_")
    out["nuclear_target_id"] = np.where(
        safe_names.astype(str).str.len() > 0,
        f"{re.sub(r'[^A-Za-z0-9]+', '_', str(prefix).lower()).strip('_') or 'agn'}:" + safe_names.astype(str),
        generated,
    )

    # Prefer explicit/name IDs, then collapse near-identical coordinate rows.
    out = out.sort_values(["nuclear_target_id", "ra", "dec"]).drop_duplicates(
        subset=["nuclear_target_id"],
        keep="first",
    )
    if drop_duplicate_coords_arcsec > 0:
        scale = float(drop_duplicate_coords_arcsec) / 3600.0
        ra_key = pd.Series(np.round(out["ra"].to_numpy(dtype=float) / scale).astype(np.int64), index=out.index)
        dec_key = pd.Series(np.round(out["dec"].to_numpy(dtype=float) / scale).astype(np.int64), index=out.index)
        coord_key = ra_key.astype(str) + "_" + dec_key.astype(str)
        out = out.assign(_coord_key=coord_key).drop_duplicates("_coord_key", keep="first")
        out = out.drop(columns=["_coord_key"])

    return out[CANONICAL_COLUMNS].reset_index(drop=True)


def load_agn_catalog(path: str | Path, *, source_catalog: str | None = None) -> pd.DataFrame:
    """Load and normalize one AGN catalog from CSV/TSV/Parquet."""
    in_path = Path(path).expanduser()
    if not in_path.exists():
        raise FileNotFoundError(f"AGN catalog not found: {in_path}")

    suffix = in_path.suffix.lower()
    if suffix == ".parquet":
        df = read_parquet_table(in_path)
    elif suffix in {".tsv", ".txt"}:
        df = pd.read_csv(in_path, sep="\t")
    else:
        df = pd.read_csv(in_path)

    source = source_catalog or in_path.stem
    return normalize_agn_catalog(df, source_catalog=source)


def build_parent_sample(
    catalog_paths: list[str | Path],
    *,
    source_catalog: str | None = None,
) -> pd.DataFrame:
    """Build a de-duplicated parent sample from one or more local catalogs."""
    frames = [
        load_agn_catalog(path, source_catalog=source_catalog)
        for path in catalog_paths
    ]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)

    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values(["nuclear_target_id", "source_catalog"]).drop_duplicates(
        subset=["nuclear_target_id"],
        keep="first",
    )
    return out.reset_index(drop=True)


def targets_from_ltv_candidates(df: pd.DataFrame) -> pd.DataFrame:
    """Create AGN targets from an existing MALCA table with MilliQuas matches."""
    if "milliquas_name" in df.columns:
        mask = df["milliquas_name"].fillna("").astype(str).str.strip() != ""
        df = df.loc[mask].copy()
    return normalize_agn_catalog(df, source_catalog="ltv_milliquas")


## Notebook-Only Source: photometry preparation

This cell contains the workflow source directly. It is no longer backed by a standalone module.

In [6]:
"""ASAS-SN light-curve preparation for AGN/nuclear variability work."""

from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from malca.io.fetch import cone_search, download_lightcurve_by_id
from malca.io.lightcurve_io import load_lightcurve_df
from malca.core.utils import clean_lc


BAND_VALUE_TO_NAME = {
    0: "g",
    1: "v",
}

MAG_TO_FLUX_ERR = np.log(10.0) / 2.5


def _band_name_from_value(value: object) -> str:
    try:
        ivalue = int(value)
    except Exception:
        return str(value).strip().lower()
    return BAND_VALUE_TO_NAME.get(ivalue, str(ivalue))


def standardize_lightcurve(df: pd.DataFrame) -> pd.DataFrame:
    """Return a MALCA-like ASAS-SN frame with normalized quality and band columns."""
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    rename_map = {
        "good/bad": "good_bad",
        "v/g?": "v_g_band",
        "saturated/unsaturated": "saturated",
        "Mag Error": "error",
        "Mag": "mag",
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})

    if "JD" not in out.columns and "jd" in out.columns:
        out["JD"] = out["jd"]
    if "error" not in out.columns and "mag_err" in out.columns:
        out["error"] = out["mag_err"]
    if "saturated" not in out.columns:
        out["saturated"] = 0
    if "good_bad" not in out.columns:
        out["good_bad"] = 1

    if "v_g_band" not in out.columns:
        if "filter_band" in out.columns:
            band = out["filter_band"].astype(str).str.strip().str.lower()
            out["v_g_band"] = band.map({"g": 0, "v": 1})
        elif "Filter" in out.columns:
            band = out["Filter"].astype(str).str.strip().str.lower()
            out["v_g_band"] = band.map({"g": 0, "v": 1})
        else:
            out["v_g_band"] = 0

    for col in ("JD", "mag", "error", "saturated", "good_bad", "v_g_band"):
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out[np.isfinite(out["JD"]) & np.isfinite(out["mag"]) & np.isfinite(out["error"])].copy()
    out = out[out["error"] > 0].copy()
    out["band"] = out["v_g_band"].map(_band_name_from_value)
    out = out.sort_values("JD").reset_index(drop=True)
    return out


def split_clean_bands(
    df: pd.DataFrame,
    *,
    max_error: float | None = None,
) -> dict[str, pd.DataFrame]:
    """Split one ASAS-SN light curve into cleaned band-specific frames."""
    norm = standardize_lightcurve(df)
    if norm.empty:
        return {}

    if max_error is not None:
        norm = norm[norm["error"] <= float(max_error)].copy()

    bands: dict[str, pd.DataFrame] = {}
    for band, band_df in norm.groupby("band", dropna=True):
        clean = clean_lc(band_df)
        if not clean.empty:
            clean["band"] = str(band)
            bands[str(band)] = clean.reset_index(drop=True)
    return bands


def mag_to_relative_flux(
    mag: np.ndarray | pd.Series,
    mag_err: np.ndarray | pd.Series,
    *,
    reference_mag: float | None = None,
) -> tuple[np.ndarray, np.ndarray, float]:
    """Convert magnitudes to relative flux with first-order error propagation."""
    mag_arr = np.asarray(mag, dtype=float)
    err_arr = np.asarray(mag_err, dtype=float)
    finite = np.isfinite(mag_arr)
    ref = float(np.nanmedian(mag_arr[finite])) if reference_mag is None else float(reference_mag)
    flux = np.power(10.0, -0.4 * (mag_arr - ref))
    flux_err = MAG_TO_FLUX_ERR * flux * err_arr
    flux[~np.isfinite(flux)] = np.nan
    flux_err[~np.isfinite(flux_err)] = np.nan
    return flux, flux_err, ref


def add_relative_flux(df: pd.DataFrame, *, reference_mag: float | None = None) -> pd.DataFrame:
    """Attach `rel_flux`, `rel_flux_err`, and `reference_mag` columns."""
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    flux, flux_err, ref = mag_to_relative_flux(out["mag"], out["error"], reference_mag=reference_mag)
    out["rel_flux"] = flux
    out["rel_flux_err"] = flux_err
    out["reference_mag"] = ref
    return out


def load_asassn_lightcurve(path: str | Path) -> pd.DataFrame:
    """Load a MALCA-supported ASAS-SN light curve path."""
    return standardize_lightcurve(load_lightcurve_df(Path(path)))


def prepare_agn_lightcurve(
    path: str | Path,
    *,
    primary_band: str = "g",
    min_points: int = 20,
    max_error: float | None = 0.5,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], dict[str, object]]:
    """Load, clean, split, and flux-normalize one AGN light curve.

    Returns `(primary_frame, all_bands, quality)` where `primary_frame` is empty
    if the requested band does not satisfy the minimum quality cuts.
    """
    raw = load_asassn_lightcurve(path)
    bands = split_clean_bands(raw, max_error=max_error)
    flux_bands = {band: add_relative_flux(frame) for band, frame in bands.items()}

    primary_key = str(primary_band).strip().lower()
    primary = flux_bands.get(primary_key, pd.DataFrame()).copy()

    quality: dict[str, object] = {
        "lc_path": str(Path(path)),
        "primary_band": primary_key,
        "n_total_points": int(len(raw)),
        "n_g_points": int(len(flux_bands.get("g", pd.DataFrame()))),
        "n_v_points": int(len(flux_bands.get("v", pd.DataFrame()))),
        "n_primary_points": int(len(primary)),
        "has_primary_band": bool(not primary.empty),
        "passes_min_points": bool(len(primary) >= int(min_points)),
    }
    if not primary.empty:
        jd = primary["JD"].to_numpy(dtype=float)
        quality.update(
            {
                "baseline_days": float(np.nanmax(jd) - np.nanmin(jd)) if len(jd) >= 2 else 0.0,
                "median_mag": float(np.nanmedian(primary["mag"])),
                "median_mag_err": float(np.nanmedian(primary["error"])),
                "reference_mag": float(np.nanmedian(primary["reference_mag"])),
            }
        )
    else:
        quality.update(
            {
                "baseline_days": 0.0,
                "median_mag": np.nan,
                "median_mag_err": np.nan,
                "reference_mag": np.nan,
            }
        )

    if len(primary) < int(min_points):
        primary = primary.iloc[0:0].copy()

    return primary, flux_bands, quality


def _match_column(df: pd.DataFrame, names: tuple[str, ...]) -> str | None:
    lower = {str(col).strip().lower(): col for col in df.columns}
    for name in names:
        key = str(name).strip().lower()
        if key in lower:
            return lower[key]
    return None


def _nearest_sky_match(matches: pd.DataFrame, ra: float, dec: float) -> tuple[pd.Series, float]:
    """Return the cone-search row nearest to the requested coordinates."""
    if matches is None or matches.empty:
        raise RuntimeError("No SkyPatrol source found within cone-search radius")

    ra_col = _match_column(matches, ("ra_deg", "ra", "RA", "RAJ2000", "raj2000"))
    dec_col = _match_column(matches, ("dec_deg", "dec", "DEC", "DEJ2000", "dej2000"))
    if ra_col is None or dec_col is None:
        return matches.iloc[0], np.nan

    match_ra = pd.to_numeric(matches[ra_col], errors="coerce").to_numpy(dtype=float)
    match_dec = pd.to_numeric(matches[dec_col], errors="coerce").to_numpy(dtype=float)
    cos_dec = np.cos(np.deg2rad(float(dec)))
    sep = 3600.0 * np.sqrt(np.square((match_ra - float(ra)) * cos_dec) + np.square(match_dec - float(dec)))
    finite = np.isfinite(sep)
    if not finite.any():
        return matches.iloc[0], np.nan
    best_pos = int(np.nanargmin(np.where(finite, sep, np.inf)))
    return matches.iloc[best_pos], float(sep[best_pos])


def resolve_or_fetch_lightcurve(
    row: pd.Series,
    *,
    cache_dir: str | Path,
    backend: str | None = None,
    refresh_cache: bool = False,
    cone_radius_arcsec: float = 3.0,
) -> tuple[str, dict[str, object]]:
    """Resolve an existing path or fetch the nearest SkyPatrol light curve."""
    for col in ("lc_path", "lightcurve_path", "asassn_lc_path"):
        value = row.get(col)
        if isinstance(value, str) and value.strip() and Path(value).expanduser().exists():
            return str(Path(value).expanduser()), {"fetch_status": "existing_path"}

    asassn_id = row.get("asassn_id")
    if pd.notna(asassn_id) and str(asassn_id).strip():
        path, meta = download_lightcurve_by_id(
            str(asassn_id).strip(),
            cache_dir=cache_dir,
            backend=backend,
            refresh_cache=refresh_cache,
        )
        out = dict(meta)
        out["fetch_status"] = "downloaded_by_asassn_id"
        out["asassn_id"] = str(asassn_id).strip()
        return str(path), out

    ra = pd.to_numeric(row.get("ra"), errors="coerce")
    dec = pd.to_numeric(row.get("dec"), errors="coerce")
    if not (np.isfinite(ra) and np.isfinite(dec)):
        raise ValueError("Cannot fetch light curve without `asassn_id` or finite RA/Dec")

    matches = cone_search(
        float(ra),
        float(dec),
        radius_arcsec=float(cone_radius_arcsec),
        catalog="master_list",
        backend=backend,
    )
    match, cone_sep_arcsec = _nearest_sky_match(matches, float(ra), float(dec))
    match_id = match.get("asas_sn_id", match.get("asassn_id", match.get("id")))
    if pd.isna(match_id):
        raise RuntimeError("Cone-search match did not include an ASAS-SN ID")

    path, meta = download_lightcurve_by_id(
        str(int(match_id)) if isinstance(match_id, (int, float, np.integer, np.floating)) else str(match_id),
        cache_dir=cache_dir,
        backend=backend,
        refresh_cache=refresh_cache,
    )
    out = dict(meta)
    out["fetch_status"] = "downloaded_by_cone_search"
    out["asassn_id"] = str(match_id)
    out["cone_sep_arcsec"] = cone_sep_arcsec
    out["cone_n_matches"] = int(len(matches))
    if "catalog_sources" in match.index:
        out["cone_catalog_sources"] = str(match.get("catalog_sources"))
    return str(path), out


def attach_lightcurve_paths(
    targets: pd.DataFrame,
    *,
    cache_dir: str | Path,
    backend: str | None = None,
    refresh_cache: bool = False,
    cone_radius_arcsec: float = 3.0,
    n_workers: int = 1,
    max_targets: int | None = None,
    verbose: bool = False,
) -> pd.DataFrame:
    """Resolve/fetch light curves for a target table and attach path/status columns."""
    work = targets.copy()
    if max_targets is not None:
        work = work.head(int(max_targets)).copy()

    rows = list(work.iterrows())
    results: list[dict[str, object]] = []

    def _one(item: tuple[object, pd.Series]) -> dict[str, object]:
        idx, row = item
        result: dict[str, object] = {"_idx": idx, "lc_path": "", "fetch_status": "failed", "fetch_error": ""}
        try:
            path, meta = resolve_or_fetch_lightcurve(
                row,
                cache_dir=cache_dir,
                backend=backend,
                refresh_cache=refresh_cache,
                cone_radius_arcsec=cone_radius_arcsec,
            )
            result.update(meta)
            result["lc_path"] = path
            result["fetch_status"] = str(meta.get("fetch_status") or "ok")
        except Exception as exc:
            result["fetch_error"] = str(exc)
        return result

    if int(n_workers) <= 1:
        iterator = rows
        if verbose:
            iterator = tqdm(iterator, total=len(rows), desc="nuclear-fetch")
        for item in iterator:
            results.append(_one(item))
    else:
        with ThreadPoolExecutor(max_workers=int(n_workers)) as executor:
            futures = [executor.submit(_one, item) for item in rows]
            iterator = as_completed(futures)
            if verbose:
                iterator = tqdm(iterator, total=len(futures), desc="nuclear-fetch")
            for future in iterator:
                results.append(future.result())

    result_df = pd.DataFrame(results).set_index("_idx") if results else pd.DataFrame()
    out = work.copy()
    for col in ("lc_path", "fetch_status", "fetch_error", "asassn_id"):
        if col not in out.columns:
            out[col] = ""
        out[col] = out[col].astype(object)
    if not result_df.empty:
        for col in result_df.columns:
            if col not in out.columns:
                out[col] = np.nan
            out[col] = out[col].astype(object)
            out.loc[result_df.index, col] = result_df[col].to_numpy(dtype=object)
    return out.reset_index(drop=True)


/opt/homebrew/Caskroom/miniconda/base/envs/malca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Notebook-Only Source: flux-domain features

This cell contains the workflow source directly. It is no longer backed by a standalone module.

In [7]:
"""Flux-domain nuclear variability features."""

from __future__ import annotations

import math
import os
import re

import numpy as np
import pandas as pd

from malca.core.stats import structure_function


FEATURE_COLUMNS = [
    "nuc_n_points",
    "nuc_baseline_days",
    "nuc_median_mag",
    "nuc_median_mag_err",
    "nuc_mean_rel_flux",
    "nuc_rel_flux_iqr",
    "nuc_rel_flux_mad",
    "nuc_excess_variance",
    "nuc_fractional_rms",
    "nuc_sf_flux_amplitude",
    "nuc_sf_flux_gamma",
    "nuc_long_slope_per_year",
    "nuc_long_slope_snr",
    "nuc_max_abs_delta_flux",
    "nuc_max_abs_delta_mag",
    "nuc_max_robust_flux_z",
    "nuc_n_transient_like_points",
    "nuc_n_seasons",
    "nuc_season_flux_std",
    "nuc_season_flux_range",
    "nuc_season_var_ratio",
    "nuc_half_mean_delta",
    "nuc_half_mean_delta_snr",
    "nuc_half_var_ratio",
    "nuc_bayesian_block_count",
]


def _nan_features() -> dict[str, float]:
    return {col: np.nan for col in FEATURE_COLUMNS}


def robust_sigma(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    return float(1.4826 * np.nanmedian(np.abs(values - np.nanmedian(values))))


def excess_variance(values: np.ndarray, errors: np.ndarray) -> tuple[float, float]:
    """Return normalized excess variance and fractional RMS."""
    y = np.asarray(values, dtype=float)
    yerr = np.asarray(errors, dtype=float)
    mask = np.isfinite(y) & np.isfinite(yerr) & (yerr >= 0)
    if mask.sum() < 2:
        return np.nan, np.nan

    y = y[mask]
    yerr = yerr[mask]
    mean = float(np.nanmean(y))
    if not np.isfinite(mean) or abs(mean) <= 0:
        return np.nan, np.nan
    sample_var = float(np.nanvar(y, ddof=1))
    noise_var = float(np.nanmean(np.square(yerr)))
    excess = max(sample_var - noise_var, 0.0) / (mean * mean)
    frac_rms = math.sqrt(excess) if excess >= 0 else np.nan
    return float(excess), float(frac_rms)


def weighted_linear_slope(
    time_days: np.ndarray,
    values: np.ndarray,
    errors: np.ndarray,
) -> tuple[float, float]:
    """Fit a weighted linear trend and return slope per year plus S/N."""
    t = np.asarray(time_days, dtype=float)
    y = np.asarray(values, dtype=float)
    yerr = np.asarray(errors, dtype=float)
    mask = np.isfinite(t) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
    if mask.sum() < 3:
        return np.nan, np.nan

    t = t[mask]
    y = y[mask]
    yerr = yerr[mask]
    x = (t - np.nanmedian(t)) / 365.25
    weights = 1.0 / np.square(yerr)
    try:
        coeff, cov = np.polyfit(x, y, deg=1, w=np.sqrt(weights), cov=True)
    except Exception:
        try:
            coeff = np.polyfit(x, y, deg=1, w=np.sqrt(weights))
            cov = None
        except Exception:
            return np.nan, np.nan
    slope = float(coeff[0])
    if cov is not None and np.shape(cov) == (2, 2) and cov[0, 0] > 0:
        slope_err = float(np.sqrt(cov[0, 0]))
        slope_snr = slope / slope_err if slope_err > 0 else np.nan
    else:
        slope_snr = np.nan
    return slope, float(slope_snr) if np.isfinite(slope_snr) else np.nan


def season_summary(
    jd: np.ndarray,
    values: np.ndarray,
    *,
    season_days: float = 365.25,
) -> dict[str, float]:
    """Compute simple year-like season statistics from a light curve."""
    t = np.asarray(jd, dtype=float)
    y = np.asarray(values, dtype=float)
    mask = np.isfinite(t) & np.isfinite(y)
    if mask.sum() < 3:
        return {
            "nuc_n_seasons": 0,
            "nuc_season_flux_std": np.nan,
            "nuc_season_flux_range": np.nan,
            "nuc_season_var_ratio": np.nan,
        }

    t = t[mask]
    y = y[mask]
    season = np.floor((t - np.nanmin(t)) / float(season_days)).astype(int)
    frame = pd.DataFrame({"season": season, "value": y})
    grouped = frame.groupby("season")["value"]
    medians = grouped.median()
    variances = grouped.var(ddof=1).replace([np.inf, -np.inf], np.nan).dropna()
    if medians.empty:
        return {
            "nuc_n_seasons": 0,
            "nuc_season_flux_std": np.nan,
            "nuc_season_flux_range": np.nan,
            "nuc_season_var_ratio": np.nan,
        }

    if len(variances) >= 2 and np.nanmin(variances) > 0:
        var_ratio = float(np.nanmax(variances) / np.nanmin(variances))
    else:
        var_ratio = np.nan

    return {
        "nuc_n_seasons": int(len(medians)),
        "nuc_season_flux_std": float(np.nanstd(medians, ddof=1)) if len(medians) >= 2 else 0.0,
        "nuc_season_flux_range": float(np.nanmax(medians) - np.nanmin(medians)),
        "nuc_season_var_ratio": var_ratio,
    }


def half_split_change(
    time_days: np.ndarray,
    values: np.ndarray,
    errors: np.ndarray,
) -> dict[str, float]:
    """Compare first and second half mean/variance as a state-change proxy."""
    t = np.asarray(time_days, dtype=float)
    y = np.asarray(values, dtype=float)
    yerr = np.asarray(errors, dtype=float)
    mask = np.isfinite(t) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
    if mask.sum() < 6:
        return {
            "nuc_half_mean_delta": np.nan,
            "nuc_half_mean_delta_snr": np.nan,
            "nuc_half_var_ratio": np.nan,
        }

    order = np.argsort(t[mask])
    y = y[mask][order]
    yerr = yerr[mask][order]
    mid = len(y) // 2
    y1, y2 = y[:mid], y[mid:]
    e1, e2 = yerr[:mid], yerr[mid:]
    if len(y1) < 3 or len(y2) < 3:
        return {
            "nuc_half_mean_delta": np.nan,
            "nuc_half_mean_delta_snr": np.nan,
            "nuc_half_var_ratio": np.nan,
        }

    mean1 = float(np.nanmean(y1))
    mean2 = float(np.nanmean(y2))
    delta = mean2 - mean1
    mean_err = math.sqrt(float(np.nanmean(np.square(e1))) / len(e1) + float(np.nanmean(np.square(e2))) / len(e2))
    snr = delta / mean_err if mean_err > 0 else np.nan
    var1 = float(np.nanvar(y1, ddof=1))
    var2 = float(np.nanvar(y2, ddof=1))
    var_ratio = max(var1, var2) / min(var1, var2) if min(var1, var2) > 0 else np.nan
    return {
        "nuc_half_mean_delta": float(delta),
        "nuc_half_mean_delta_snr": float(snr) if np.isfinite(snr) else np.nan,
        "nuc_half_var_ratio": float(var_ratio) if np.isfinite(var_ratio) else np.nan,
    }


def bayesian_block_count(time_days: np.ndarray, values: np.ndarray, errors: np.ndarray) -> float:
    """Return the number of Bayesian-block segments for a rough change-point score."""
    try:
        from astropy.stats import bayesian_blocks
    except Exception:
        return np.nan

    t = np.asarray(time_days, dtype=float)
    y = np.asarray(values, dtype=float)
    yerr = np.asarray(errors, dtype=float)
    mask = np.isfinite(t) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
    if mask.sum() < 8:
        return np.nan
    try:
        edges = bayesian_blocks(t[mask], x=y[mask], sigma=yerr[mask], fitness="measures")
    except Exception:
        return np.nan
    return float(max(len(edges) - 1, 0))


def compute_flux_features(df: pd.DataFrame) -> dict[str, float]:
    """Compute AGN variability features from a prepared primary-band frame."""
    out = _nan_features()
    if df is None or df.empty:
        return out

    required = {"JD", "mag", "error", "rel_flux", "rel_flux_err"}
    if missing := required - set(df.columns):
        raise ValueError(f"Prepared AGN light curve missing columns: {sorted(missing)}")

    jd = df["JD"].to_numpy(dtype=float)
    mag = df["mag"].to_numpy(dtype=float)
    mag_err = df["error"].to_numpy(dtype=float)
    flux = df["rel_flux"].to_numpy(dtype=float)
    flux_err = df["rel_flux_err"].to_numpy(dtype=float)

    mask = (
        np.isfinite(jd)
        & np.isfinite(mag)
        & np.isfinite(mag_err)
        & np.isfinite(flux)
        & np.isfinite(flux_err)
        & (mag_err > 0)
        & (flux_err > 0)
    )
    if mask.sum() < 2:
        return out

    jd = jd[mask]
    mag = mag[mask]
    mag_err = mag_err[mask]
    flux = flux[mask]
    flux_err = flux_err[mask]
    baseline = float(np.nanmax(jd) - np.nanmin(jd)) if len(jd) >= 2 else 0.0
    median_flux = float(np.nanmedian(flux))
    flux_resid = flux - median_flux
    sigma_robust = robust_sigma(flux_resid)
    sigma_eff = math.sqrt(sigma_robust * sigma_robust + float(np.nanmedian(np.square(flux_err)))) if np.isfinite(sigma_robust) else np.nan
    robust_z = np.abs(flux_resid) / sigma_eff if np.isfinite(sigma_eff) and sigma_eff > 0 else np.full_like(flux, np.nan)

    exvar, frac_rms = excess_variance(flux, flux_err)
    sf_amp, sf_gamma = structure_function(flux, jd)
    slope, slope_snr = weighted_linear_slope(jd, flux, flux_err)

    out.update(
        {
            "nuc_n_points": int(len(jd)),
            "nuc_baseline_days": baseline,
            "nuc_median_mag": float(np.nanmedian(mag)),
            "nuc_median_mag_err": float(np.nanmedian(mag_err)),
            "nuc_mean_rel_flux": float(np.nanmean(flux)),
            "nuc_rel_flux_iqr": float(np.nanpercentile(flux, 75) - np.nanpercentile(flux, 25)),
            "nuc_rel_flux_mad": sigma_robust,
            "nuc_excess_variance": exvar,
            "nuc_fractional_rms": frac_rms,
            "nuc_sf_flux_amplitude": float(sf_amp) if np.isfinite(sf_amp) else np.nan,
            "nuc_sf_flux_gamma": float(sf_gamma) if np.isfinite(sf_gamma) else np.nan,
            "nuc_long_slope_per_year": slope,
            "nuc_long_slope_snr": slope_snr,
            "nuc_max_abs_delta_flux": float(np.nanmax(np.abs(flux_resid))),
            "nuc_max_abs_delta_mag": float(np.nanmax(np.abs(mag - np.nanmedian(mag)))),
            "nuc_max_robust_flux_z": float(np.nanmax(robust_z)) if np.isfinite(robust_z).any() else np.nan,
            "nuc_n_transient_like_points": int(np.nansum(robust_z >= 5.0)) if np.isfinite(robust_z).any() else 0,
            "nuc_bayesian_block_count": bayesian_block_count(jd, flux, flux_err),
        }
    )
    out.update(season_summary(jd, flux))
    out.update(half_split_change(jd, flux, flux_err))
    return out


## Notebook-Only Source: stochastic metrics

This cell contains the workflow source directly. It is no longer backed by a standalone module.

In [8]:
"""AGN stochastic-model features and reliability diagnostics."""

from __future__ import annotations

import numpy as np
import pandas as pd

from malca.core.stats import fit_drw, iar_phi_fit, mhps, structure_function


STOCHASTIC_COLUMNS = [
    "nuc_stoch_sf_amplitude",
    "nuc_stoch_sf_gamma",
    "nuc_stoch_iar_phi",
    "nuc_stoch_mhps_high",
    "nuc_stoch_mhps_low",
    "nuc_stoch_mhps_non_zero",
    "nuc_stoch_mhps_pn_flag",
    "nuc_stoch_mhps_ratio",
    "nuc_drw_sigma",
    "nuc_drw_tau_days",
    "nuc_drw_baseline_tau_ratio",
    "nuc_drw_tau_fraction_of_baseline",
    "nuc_drw_reliable",
    "nuc_drw_tau_at_boundary",
    "nuc_stoch_faint_noisy",
    "nuc_stoch_seasonal_gap_risk",
    "nuc_stoch_reliability_score",
]


def _empty_result() -> dict[str, object]:
    out: dict[str, object] = {}
    for col in STOCHASTIC_COLUMNS:
        if col in {"nuc_drw_reliable", "nuc_drw_tau_at_boundary", "nuc_stoch_faint_noisy", "nuc_stoch_seasonal_gap_risk"}:
            out[col] = False
        else:
            out[col] = np.nan
    return out


def sampling_diagnostics(jd: np.ndarray) -> dict[str, float]:
    """Return baseline/cadence diagnostics for irregular ASAS-SN sampling."""
    t = np.asarray(jd, dtype=float)
    t = np.sort(t[np.isfinite(t)])
    if t.size < 2:
        return {
            "baseline_days": 0.0,
            "median_cadence_days": np.nan,
            "max_gap_days": np.nan,
            "max_gap_fraction": np.nan,
        }
    gaps = np.diff(t)
    baseline = float(t[-1] - t[0])
    max_gap = float(np.nanmax(gaps)) if gaps.size else np.nan
    return {
        "baseline_days": baseline,
        "median_cadence_days": float(np.nanmedian(gaps)) if gaps.size else np.nan,
        "max_gap_days": max_gap,
        "max_gap_fraction": max_gap / baseline if baseline > 0 and np.isfinite(max_gap) else np.nan,
    }


def drw_reliability_flags(
    *,
    n_points: int,
    baseline_days: float,
    tau_days: float,
    median_mag: float | None = None,
    median_mag_err: float | None = None,
    max_gap_fraction: float | None = None,
) -> dict[str, object]:
    """Evaluate whether an ASAS-SN DRW timescale should be trusted."""
    tau = float(tau_days) if np.isfinite(tau_days) else np.nan
    baseline = float(baseline_days) if np.isfinite(baseline_days) else 0.0
    ratio = baseline / tau if baseline > 0 and np.isfinite(tau) and tau > 0 else np.nan
    tau_fraction = tau / baseline if baseline > 0 and np.isfinite(tau) else np.nan

    tau_at_boundary = bool(np.isfinite(tau_fraction) and tau_fraction >= 0.5)
    faint_noisy = bool(
        (median_mag is not None and np.isfinite(median_mag) and median_mag >= 17.0)
        or (median_mag_err is not None and np.isfinite(median_mag_err) and median_mag_err >= 0.15)
    )
    seasonal_gap_risk = bool(max_gap_fraction is not None and np.isfinite(max_gap_fraction) and max_gap_fraction >= 0.25)
    reliable = bool(
        int(n_points) >= 50
        and np.isfinite(ratio)
        and ratio >= 10.0
        and not tau_at_boundary
        and not faint_noisy
    )

    score = 0.0
    if int(n_points) >= 50:
        score += 0.25
    if int(n_points) >= 100:
        score += 0.15
    if np.isfinite(ratio):
        score += min(max(ratio / 20.0, 0.0), 0.35)
    if not faint_noisy:
        score += 0.15
    if not seasonal_gap_risk:
        score += 0.10

    return {
        "nuc_drw_baseline_tau_ratio": float(ratio) if np.isfinite(ratio) else np.nan,
        "nuc_drw_tau_fraction_of_baseline": float(tau_fraction) if np.isfinite(tau_fraction) else np.nan,
        "nuc_drw_reliable": reliable,
        "nuc_drw_tau_at_boundary": tau_at_boundary,
        "nuc_stoch_faint_noisy": faint_noisy,
        "nuc_stoch_seasonal_gap_risk": seasonal_gap_risk,
        "nuc_stoch_reliability_score": float(min(score, 1.0)),
    }


def compute_stochastic_features(
    df: pd.DataFrame,
    *,
    include_drw: bool = True,
) -> dict[str, object]:
    """Compute stochastic features on relative flux for a prepared AGN light curve."""
    out = _empty_result()
    if df is None or df.empty:
        return out

    required = {"JD", "rel_flux", "rel_flux_err", "mag", "error"}
    if missing := required - set(df.columns):
        raise ValueError(f"Prepared AGN light curve missing columns: {sorted(missing)}")

    jd = df["JD"].to_numpy(dtype=float)
    flux = df["rel_flux"].to_numpy(dtype=float)
    flux_err = df["rel_flux_err"].to_numpy(dtype=float)
    mag = df["mag"].to_numpy(dtype=float)
    mag_err = df["error"].to_numpy(dtype=float)
    mask = np.isfinite(jd) & np.isfinite(flux) & np.isfinite(flux_err) & (flux_err > 0)
    if mask.sum() < 10:
        return out

    jd = jd[mask]
    flux = flux[mask]
    flux_err = flux_err[mask]
    diag = sampling_diagnostics(jd)

    sf_amp, sf_gamma = structure_function(flux, jd)
    out["nuc_stoch_sf_amplitude"] = float(sf_amp) if np.isfinite(sf_amp) else np.nan
    out["nuc_stoch_sf_gamma"] = float(sf_gamma) if np.isfinite(sf_gamma) else np.nan

    iar_phi = iar_phi_fit(jd, flux, flux_err)
    out["nuc_stoch_iar_phi"] = float(iar_phi) if np.isfinite(iar_phi) else np.nan

    mhps_result = mhps(jd, flux, flux_err)
    for key in ("mhps_high", "mhps_low", "mhps_non_zero", "mhps_pn_flag", "mhps_ratio"):
        value = mhps_result.get(key, np.nan)
        out[f"nuc_stoch_{key}"] = float(value) if np.isfinite(value) else np.nan

    if include_drw:
        sigma, tau = fit_drw(jd, flux, flux_err)
        out["nuc_drw_sigma"] = float(sigma) if np.isfinite(sigma) else np.nan
        out["nuc_drw_tau_days"] = float(tau) if np.isfinite(tau) else np.nan
    else:
        tau = np.nan

    out.update(
        drw_reliability_flags(
            n_points=int(mask.sum()),
            baseline_days=diag["baseline_days"],
            tau_days=float(out["nuc_drw_tau_days"]) if np.isfinite(out["nuc_drw_tau_days"]) else np.nan,
            median_mag=float(np.nanmedian(mag)) if np.isfinite(mag).any() else None,
            median_mag_err=float(np.nanmedian(mag_err)) if np.isfinite(mag_err).any() else None,
            max_gap_fraction=diag["max_gap_fraction"],
        )
    )
    return out


## Notebook-Only Source: staged pipeline

This cell contains the workflow source directly. It is no longer backed by a standalone module.

In [9]:
"""Staged all-sky AGN stochastic variability pipeline."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Notebook-only: build_parent_sample, compute_flux_features, attach_lightcurve_paths,
# prepare_agn_lightcurve, and compute_stochastic_features are defined in earlier cells.
from malca.io.table_io import read_parquet_table, write_parquet_table


DEFAULT_RUN_DIR = Path("output/runs/agn_stochastic")

TARGETS_FILENAME = "parent_agn_targets.parquet"
TARGETS_WITH_LCS_FILENAME = "parent_agn_targets_with_lightcurves.parquet"
QUALITY_FILENAME = "lightcurve_quality.parquet"
FEATURES_FILENAME = "stochastic_features.parquet"
RANKED_FILENAME = "ranked_outliers.parquet"
REVIEW_FILENAME = "review_candidates.parquet"


def targets_path(run_dir: str | Path = DEFAULT_RUN_DIR) -> Path:
    return Path(run_dir) / TARGETS_FILENAME


def targets_with_lcs_path(run_dir: str | Path = DEFAULT_RUN_DIR) -> Path:
    return Path(run_dir) / TARGETS_WITH_LCS_FILENAME


def quality_path(run_dir: str | Path = DEFAULT_RUN_DIR) -> Path:
    return Path(run_dir) / QUALITY_FILENAME


def features_path(run_dir: str | Path = DEFAULT_RUN_DIR) -> Path:
    return Path(run_dir) / FEATURES_FILENAME


def ranked_path(run_dir: str | Path = DEFAULT_RUN_DIR) -> Path:
    return Path(run_dir) / RANKED_FILENAME


def review_path(run_dir: str | Path = DEFAULT_RUN_DIR) -> Path:
    return Path(run_dir) / REVIEW_FILENAME


def _read_table(path: str | Path) -> pd.DataFrame:
    return read_parquet_table(Path(path))


def _write_table(df: pd.DataFrame, path: str | Path) -> None:
    write_parquet_table(df, Path(path))


def build_targets_stage(
    catalog_paths: list[str | Path],
    *,
    output: str | Path | None = None,
    run_dir: str | Path = DEFAULT_RUN_DIR,
    source_catalog: str | None = None,
) -> pd.DataFrame:
    targets = build_parent_sample(catalog_paths, source_catalog=source_catalog)
    _write_table(targets, output or targets_path(run_dir))
    return targets


def fetch_lightcurves_stage(
    *,
    targets: str | Path | pd.DataFrame | None = None,
    output: str | Path | None = None,
    run_dir: str | Path = DEFAULT_RUN_DIR,
    cache_dir: str | Path | None = None,
    backend: str | None = None,
    refresh_cache: bool = False,
    cone_radius_arcsec: float = 3.0,
    max_targets: int | None = None,
    n_workers: int = 1,
    verbose: bool = True,
) -> pd.DataFrame:
    target_df = targets if isinstance(targets, pd.DataFrame) else _read_table(targets or targets_path(run_dir))
    lc_cache = Path(cache_dir) if cache_dir is not None else Path(run_dir) / "lightcurves"
    out = attach_lightcurve_paths(
        target_df,
        cache_dir=lc_cache,
        backend=backend,
        refresh_cache=refresh_cache,
        cone_radius_arcsec=cone_radius_arcsec,
        max_targets=max_targets,
        n_workers=n_workers,
        verbose=verbose,
    )
    _write_table(out, output or targets_with_lcs_path(run_dir))
    return out


def _clean_payload_value(value: object) -> object:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def _row_identity(row: pd.Series) -> dict[str, object]:
    keys = [
        "nuclear_target_id",
        "source_catalog",
        "source_name",
        "ra",
        "dec",
        "agn_type",
        "redshift",
        "asassn_id",
        "gaia_id",
        "lc_path",
    ]
    return {key: _clean_payload_value(row.get(key)) for key in keys if key in row.index}


def compute_one_target(
    row: pd.Series,
    *,
    primary_band: str = "g",
    min_points: int = 20,
    max_error: float | None = 0.5,
    include_drw: bool = True,
) -> tuple[dict[str, object], dict[str, object]]:
    identity = _row_identity(row)
    feature_row: dict[str, object] = dict(identity)
    quality_row: dict[str, object] = dict(identity)
    path = row.get("lc_path")
    if not isinstance(path, str) or not path.strip():
        feature_row["nuc_feature_status"] = "missing_lc_path"
        quality_row["quality_status"] = "missing_lc_path"
        quality_row["quality_error"] = ""
        return feature_row, quality_row

    try:
        primary, _bands, quality = prepare_agn_lightcurve(
            path,
            primary_band=primary_band,
            min_points=min_points,
            max_error=max_error,
        )
        quality_row.update(quality)
        quality_row["quality_status"] = "ok" if not primary.empty else "failed_quality"
        quality_row["quality_error"] = ""

        feature_row.update(quality)
        if primary.empty:
            feature_row["nuc_feature_status"] = "failed_quality"
            return feature_row, quality_row

        feature_row.update(compute_flux_features(primary))
        feature_row.update(compute_stochastic_features(primary, include_drw=include_drw))
        feature_row["nuc_feature_status"] = "ok"
    except Exception as exc:
        feature_row["nuc_feature_status"] = "error"
        feature_row["nuc_feature_error"] = str(exc)
        quality_row["quality_status"] = "error"
        quality_row["quality_error"] = str(exc)
    return feature_row, quality_row


def compute_features_stage(
    *,
    targets: str | Path | pd.DataFrame | None = None,
    features_output: str | Path | None = None,
    quality_output: str | Path | None = None,
    run_dir: str | Path = DEFAULT_RUN_DIR,
    primary_band: str = "g",
    min_points: int = 20,
    max_error: float | None = 0.5,
    include_drw: bool = True,
    max_targets: int | None = None,
    verbose: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_df = targets if isinstance(targets, pd.DataFrame) else _read_table(targets or targets_with_lcs_path(run_dir))
    if max_targets is not None:
        target_df = target_df.head(int(max_targets)).copy()

    feature_rows: list[dict[str, object]] = []
    quality_rows: list[dict[str, object]] = []
    iterator: Iterable[tuple[object, pd.Series]] = target_df.iterrows()
    if verbose:
        iterator = tqdm(list(iterator), desc="nuclear-features")

    for _idx, row in iterator:
        feature_row, quality_row = compute_one_target(
            row,
            primary_band=primary_band,
            min_points=min_points,
            max_error=max_error,
            include_drw=include_drw,
        )
        feature_rows.append(feature_row)
        quality_rows.append(quality_row)

    features = pd.DataFrame(feature_rows)
    quality = pd.DataFrame(quality_rows)
    _write_table(features, features_output or features_path(run_dir))
    _write_table(quality, quality_output or quality_path(run_dir))
    return features, quality


def _robust_positive_z(values: pd.Series) -> pd.Series:
    x = pd.to_numeric(values, errors="coerce").astype(float)
    finite = np.isfinite(x)
    out = pd.Series(np.nan, index=values.index, dtype=float)
    if finite.sum() < 3:
        return out
    med = float(np.nanmedian(x[finite]))
    mad = float(1.4826 * np.nanmedian(np.abs(x[finite] - med)))
    if not np.isfinite(mad) or mad <= 0:
        std = float(np.nanstd(x[finite], ddof=1))
        mad = std if std > 0 else np.nan
    if np.isfinite(mad) and mad > 0:
        out.loc[finite] = np.maximum((x[finite] - med) / mad, 0.0)
    return out


def _binned_robust_positive_z(
    df: pd.DataFrame,
    value_col: str,
    *,
    min_bin_size: int = 20,
) -> pd.Series:
    if value_col not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)

    mag = pd.to_numeric(df.get("nuc_median_mag"), errors="coerce")
    redshift = pd.to_numeric(df.get("redshift"), errors="coerce")
    mag_bin = pd.cut(mag, bins=[-np.inf, 14, 15, 16, 17, 18, np.inf], labels=False)
    z_bin = pd.cut(redshift, bins=[-np.inf, 0.03, 0.1, 0.3, 1.0, np.inf], labels=False)
    group_key = mag_bin.astype("Int64").astype(str) + "_" + z_bin.astype("Int64").astype(str)
    out = pd.Series(np.nan, index=df.index, dtype=float)
    global_z = _robust_positive_z(df[value_col])

    for key in sorted(group_key.dropna().unique()):
        idx = group_key[group_key == key].index
        if len(idx) < min_bin_size:
            continue
        out.loc[idx] = _robust_positive_z(df.loc[idx, value_col])
    out = out.fillna(global_z)
    return out


def _numeric_column(df: pd.DataFrame, col: str, default: float = 0.0) -> pd.Series:
    if col not in df.columns:
        return pd.Series(default, index=df.index, dtype=float)
    return pd.to_numeric(df[col], errors="coerce").fillna(default).astype(float)


def _bool_column(df: pd.DataFrame, col: str, default: bool = False) -> pd.Series:
    if col not in df.columns:
        return pd.Series(default, index=df.index, dtype=bool)
    return df[col].fillna(default).astype(bool)


def _bounded_score(values, *, max_score: float = 5.0) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=max_score, neginf=0.0)
    return np.clip(arr, 0.0, float(max_score))


def _snr_component(values, *, snr_unit: float = 4.0, max_score: float = 5.0) -> np.ndarray:
    arr = np.abs(np.asarray(values, dtype=float)) / float(snr_unit)
    return _bounded_score(arr, max_score=max_score)


def rank_outliers_stage(
    *,
    features: str | Path | pd.DataFrame | None = None,
    output: str | Path | None = None,
    run_dir: str | Path = DEFAULT_RUN_DIR,
) -> pd.DataFrame:
    df = features if isinstance(features, pd.DataFrame) else _read_table(features or features_path(run_dir))
    if df.empty:
        _write_table(df, output or ranked_path(run_dir))
        return df

    out = df.copy()
    ok = pd.Series(True, index=out.index)
    if "nuc_feature_status" in out.columns:
        ok = out["nuc_feature_status"].fillna("").astype(str) == "ok"

    out["nuc_excess_variance_z"] = _binned_robust_positive_z(out, "nuc_excess_variance")
    out["nuc_sf_amplitude_z"] = _binned_robust_positive_z(out, "nuc_sf_flux_amplitude")
    out["nuc_drw_sigma_z"] = _binned_robust_positive_z(out, "nuc_drw_sigma")

    stochastic_score = np.nanmax(
        np.vstack(
            [
                _bounded_score(out["nuc_excess_variance_z"].fillna(0).to_numpy(dtype=float)),
                _bounded_score(out["nuc_sf_amplitude_z"].fillna(0).to_numpy(dtype=float)),
                _bounded_score(out["nuc_drw_sigma_z"].fillna(0).to_numpy(dtype=float)),
            ]
        ),
        axis=0,
    )
    state_score = np.nanmax(
        np.vstack(
            [
                _snr_component(_numeric_column(out, "nuc_long_slope_snr").to_numpy(dtype=float)),
                _snr_component(_numeric_column(out, "nuc_half_mean_delta_snr").to_numpy(dtype=float)),
                _bounded_score(np.log1p(_numeric_column(out, "nuc_half_var_ratio").clip(lower=0).to_numpy(dtype=float)) / np.log(5.0)),
                _bounded_score(_numeric_column(out, "nuc_bayesian_block_count").clip(lower=1).to_numpy(dtype=float) - 1.0),
            ]
        ),
        axis=0,
    )
    transient_score = _bounded_score(
        _numeric_column(out, "nuc_max_robust_flux_z").to_numpy(dtype=float) / 5.0
        + np.minimum(_numeric_column(out, "nuc_n_transient_like_points").to_numpy(dtype=float), 5.0) / 5.0
    )
    drw_interest = _bounded_score(
        _bool_column(out, "nuc_drw_tau_at_boundary").to_numpy(dtype=float)
        + out["nuc_drw_sigma_z"].fillna(0).to_numpy(dtype=float) / 3.0
    )

    out["nuc_stochastic_outlier_score"] = stochastic_score
    out["nuc_state_change_score"] = state_score
    out["nuc_transient_like_score"] = transient_score
    out["nuc_drw_interest_score"] = drw_interest
    out["nuc_outlier_score"] = (
        1.0 * stochastic_score
        + 1.2 * state_score
        + 1.4 * transient_score
        + 0.7 * drw_interest
        + 0.5 * _numeric_column(out, "nuc_stoch_reliability_score").to_numpy(dtype=float)
    )
    out.loc[~ok, "nuc_outlier_score"] = np.nan

    labels = np.full(len(out), "agn_variability_candidate", dtype=object)
    labels[stochastic_score >= 3.0] = "stochastic_outlier"
    labels[drw_interest >= 1.0] = "drw_timescale_edge"
    labels[state_score >= 1.5] = "state_change_candidate"
    labels[transient_score >= 1.5] = "transient_like_excursion"
    labels[~ok.to_numpy(dtype=bool)] = "not_ranked"
    out["nuc_outlier_label"] = labels

    out = out.sort_values("nuc_outlier_score", ascending=False, na_position="last").reset_index(drop=True)
    out["nuc_rank"] = np.arange(1, len(out) + 1)
    _write_table(out, output or ranked_path(run_dir))
    return out


def export_review_stage(
    *,
    ranked: str | Path | pd.DataFrame | None = None,
    output: str | Path | None = None,
    run_dir: str | Path = DEFAULT_RUN_DIR,
    min_score: float = 2.0,
    max_candidates: int | None = None,
) -> pd.DataFrame:
    ranked_df = ranked if isinstance(ranked, pd.DataFrame) else _read_table(ranked or ranked_path(run_dir))
    if ranked_df.empty:
        review = pd.DataFrame()
        _write_table(review, output or review_path(run_dir))
        return review

    score = pd.to_numeric(ranked_df.get("nuc_outlier_score"), errors="coerce")
    review = ranked_df.loc[score >= float(min_score)].copy()
    if max_candidates is not None:
        review = review.head(int(max_candidates)).copy()

    if review.empty:
        _write_table(review, output or review_path(run_dir))
        return review

    review["candidate_id"] = review["nuclear_target_id"].astype(str)
    review["event_class"] = "nuclear_agn_outlier"
    review["interest_score"] = pd.to_numeric(review["nuc_outlier_score"], errors="coerce")
    review["disposition"] = "unreviewed"
    review["morphology_primary"] = review["nuc_outlier_label"].astype(str)
    review["physical_primary"] = "agn_or_nuclear_variability"
    review["review_notes"] = (
        "AGN stochastic outlier: "
        + review["nuc_outlier_label"].astype(str)
        + "; score="
        + review["interest_score"].map(lambda x: f"{x:.2f}" if np.isfinite(x) else "nan")
    )

    payload_cols = [
        "source_catalog",
        "source_name",
        "agn_type",
        "redshift",
        "asassn_id",
        "gaia_id",
        "lc_path",
        "nuc_outlier_label",
        "nuc_outlier_score",
        "nuc_drw_tau_days",
        "nuc_drw_reliable",
        "nuc_stochastic_outlier_score",
        "nuc_state_change_score",
        "nuc_transient_like_score",
    ]

    def _payload(row: pd.Series) -> str:
        payload = {
            col: _clean_payload_value(row.get(col))
            for col in payload_cols
            if col in row.index
        }
        return json.dumps(payload, sort_keys=True)

    review["payload_json"] = review.apply(_payload, axis=1)
    columns = [
        "candidate_id",
        "ra",
        "dec",
        "event_class",
        "interest_score",
        "disposition",
        "morphology_primary",
        "physical_primary",
        "review_notes",
        "payload_json",
    ]
    keep = columns + [col for col in review.columns if col not in columns]
    review = review[keep].reset_index(drop=True)
    _write_table(review, output or review_path(run_dir))
    return review


def run_all_stage(
    catalog_paths: list[str | Path],
    *,
    run_dir: str | Path = DEFAULT_RUN_DIR,
    source_catalog: str | None = None,
    cache_dir: str | Path | None = None,
    backend: str | None = None,
    refresh_cache: bool = False,
    cone_radius_arcsec: float = 3.0,
    primary_band: str = "g",
    min_points: int = 20,
    max_error: float | None = 0.5,
    include_drw: bool = True,
    max_targets: int | None = None,
    n_workers: int = 1,
    verbose: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    targets = build_targets_stage(catalog_paths, run_dir=run_dir, source_catalog=source_catalog)
    targets_lc = fetch_lightcurves_stage(
        targets=targets,
        run_dir=run_dir,
        cache_dir=cache_dir,
        backend=backend,
        refresh_cache=refresh_cache,
        cone_radius_arcsec=cone_radius_arcsec,
        max_targets=max_targets,
        n_workers=n_workers,
        verbose=verbose,
    )
    features, quality = compute_features_stage(
        targets=targets_lc,
        run_dir=run_dir,
        primary_band=primary_band,
        min_points=min_points,
        max_error=max_error,
        include_drw=include_drw,
        verbose=verbose,
    )
    ranked = rank_outliers_stage(features=features, run_dir=run_dir)
    review = export_review_stage(ranked=ranked, run_dir=run_dir)
    return targets_lc, features, quality, ranked, review


## Run The Pipeline Stages In Notebook

Flip individual `RUN_*` flags to execute the staged workflow on your configured `CATALOG_PATHS` and `RUN_DIR`.

In [10]:
# Notebook-native version of the staged workflow.
# Set CATALOG_PATHS in the setup cell before running this on a real sample.

# Notebook-only: stage functions are defined in the inline staged-pipeline cell above.

RUN_BUILD_TARGETS = False
RUN_FETCH_LIGHTCURVES = False
RUN_COMPUTE_FEATURES = False
RUN_RANK_OUTLIERS = False
RUN_EXPORT_REVIEW = False

if RUN_BUILD_TARGETS:
    targets = build_targets_stage(CATALOG_PATHS, run_dir=RUN_DIR)
    display(targets.head())

if RUN_FETCH_LIGHTCURVES:
    targets_lc = fetch_lightcurves_stage(
        run_dir=RUN_DIR,
        cache_dir=RUN_DIR / 'lightcurves',
        cone_radius_arcsec=3.0,
        n_workers=1,
        verbose=True,
    )
    display(targets_lc.head())

if RUN_COMPUTE_FEATURES:
    features, quality = compute_features_stage(
        run_dir=RUN_DIR,
        primary_band='g',
        min_points=20,
        max_error=0.5,
        include_drw=True,
        verbose=True,
    )
    display(quality.head())
    display(features.head())

if RUN_RANK_OUTLIERS:
    ranked = rank_outliers_stage(run_dir=RUN_DIR)
    display(ranked.head(25))

if RUN_EXPORT_REVIEW:
    review = export_review_stage(run_dir=RUN_DIR, min_score=2.0)
    display(review.head(25))


## Local Synthetic Demo

This creates three toy AGN light curves locally, builds a parent catalog with `lc_path` values, computes features, ranks outliers, and exports review-ready rows. It does not fetch from SkyPatrol.

In [11]:
# Fully local synthetic demo: no SkyPatrol/network needed.
# This exercises build-targets -> compute-features -> rank-outliers -> export-review.

import numpy as np
import pandas as pd

# Notebook-only: stage functions are defined in the inline staged-pipeline cell above.

DEMO_DIR = REPO_ROOT / 'output' / 'runs' / 'agn_stochastic_notebook_demo'
DEMO_DIR.mkdir(parents=True, exist_ok=True)


def _prepared_demo_frame(kind: str = 'quiet', n: int = 60) -> pd.DataFrame:
    jd = 2450000.0 + np.arange(n, dtype=float) * 10.0
    flux = np.ones(n, dtype=float)
    if kind == 'ramp':
        flux = 1.0 + np.linspace(0.0, 0.35, n)
    elif kind == 'flare':
        flux[30] += 1.0
    elif kind == 'drwish':
        rng = np.random.default_rng(42)
        flux = 1.0 + np.cumsum(rng.normal(0, 0.01, n))
    flux_err = np.full(n, 0.02)
    mag = 15.0 - 2.5 * np.log10(flux)
    return pd.DataFrame(
        {
            'JD': jd,
            'Flux': flux,
            'Flux Error': flux_err,
            'Mag': mag,
            'Mag Error': 0.02,
            'Limit': 18.5,
            'FWHM': 2.0,
            'Filter': 'g',
            'Quality': 'G',
            'Camera': 'cam1',
        }
    )

quiet_lc = DEMO_DIR / 'quiet_agn.csv'
flare_lc = DEMO_DIR / 'flare_agn.csv'
ramp_lc = DEMO_DIR / 'ramp_agn.csv'
_prepared_demo_frame('quiet').to_csv(quiet_lc, index=False)
_prepared_demo_frame('flare').to_csv(flare_lc, index=False)
_prepared_demo_frame('ramp').to_csv(ramp_lc, index=False)

catalog_path = DEMO_DIR / 'demo_agn_catalog.csv'
pd.DataFrame(
    {
        'Name': ['Quiet AGN', 'Flare AGN', 'Ramp AGN'],
        'RAJ2000': [1.0, 2.0, 3.0],
        'DEJ2000': [3.0, 4.0, 5.0],
        'Type': ['QSO', 'Seyfert', 'QSO'],
        'z': [0.1, 0.03, 0.2],
        'gmag': [15.0, 15.0, 15.0],
        'lc_path': [str(quiet_lc), str(flare_lc), str(ramp_lc)],
    }
).to_csv(catalog_path, index=False)

targets = build_targets_stage([catalog_path], run_dir=DEMO_DIR)
features, quality = compute_features_stage(
    targets=targets,
    run_dir=DEMO_DIR,
    min_points=20,
    include_drw=False,
    verbose=False,
)
ranked = rank_outliers_stage(features=features, run_dir=DEMO_DIR)
review = export_review_stage(ranked=ranked, run_dir=DEMO_DIR, min_score=0.0)

display(ranked[[
    'nuc_rank',
    'source_name',
    'nuc_outlier_label',
    'nuc_outlier_score',
    'nuc_state_change_score',
    'nuc_transient_like_score',
    'nuc_max_robust_flux_z',
    'nuc_long_slope_snr',
]])
display(review[['candidate_id', 'event_class', 'interest_score', 'morphology_primary', 'review_notes']])


,nuc_rank,source_name,nuc_outlier_label,nuc_outlier_score,nuc_state_change_score,nuc_transient_like_score,nuc_max_robust_flux_z,nuc_long_slope_snr
0,1,Ramp AGN,state_change_candidate,11.616520,5.0,0.2618,1.308998,5.768965e+14
1,2,Flare AGN,transient_like_excursion,10.407565,2.0,5.0000,54.286810,1.440331e-02
2,3,Quiet AGN,agn_variability_candidate,0.250000,0.0,0.0000,0.000000,0.000000e+00


,candidate_id,event_class,interest_score,morphology_primary,review_notes
0,demo_agn_catalog:Ramp_AGN,nuclear_agn_outlier,11.616520,state_change_candidate,AGN stochastic outlier: state_change_candidate...
1,demo_agn_catalog:Flare_AGN,nuclear_agn_outlier,10.407565,transient_like_excursion,AGN stochastic outlier: transient_like_excursi...
2,demo_agn_catalog:Quiet_AGN,nuclear_agn_outlier,0.250000,agn_variability_candidate,AGN stochastic outlier: agn_variability_candid...


## Real AGN Mini-Run

This section runs the same notebook-only pipeline on a small curated set of real AGN. It fetches ASAS-SN/SkyPatrol light curves by cone search, writes a real-run catalog and parquet outputs, computes stochastic features with DRW enabled, ranks outliers, and exports review-ready rows.


In [12]:
# Real AGN mini-run: fetch SkyPatrol light curves and run the full pipeline.
# The synthetic demo above stays untouched; this writes into a separate run directory.

import numpy as np
import pandas as pd

REAL_AGN_DIR = REPO_ROOT / 'output' / 'runs' / 'agn_stochastic_real_agn'
REAL_AGN_DIR.mkdir(parents=True, exist_ok=True)

REAL_AGN_BACKEND = 'skypatrol2'
REAL_AGN_FALLBACK_BACKEND = 'skypatrol1'
REAL_AGN_CONE_RADIUS_ARCSEC = 20.0
REAL_AGN_PRIMARY_BAND = 'g'
REAL_AGN_FALLBACK_BAND = 'v'


def _coerce_real_run_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Keep mixed ID/status columns parquet-safe after backend fallback merges."""
    text_cols = {
        'nuclear_target_id',
        'source_catalog',
        'source_name',
        'agn_type',
        'asassn_id',
        'gaia_id',
        'lc_path',
        'fetch_status',
        'fetch_error',
        'cone_catalog_sources',
        'nuc_feature_status',
        'nuc_feature_error',
        'quality_status',
        'quality_error',
        'primary_band',
        'primary_band_used',
    }

    def _clean_text_value(value: object) -> str:
        if value is None:
            return ''
        try:
            missing = pd.isna(value)
            if isinstance(missing, (bool, np.bool_)) and missing:
                return ''
        except Exception:
            pass
        return str(value)

    rebuilt = []
    for pos, col in enumerate(df.columns):
        series = df.iloc[:, pos]
        if col in text_cols or series.dtype == object or str(series.dtype).startswith('string'):
            series = series.map(_clean_text_value).astype('string')
        rebuilt.append(series.reset_index(drop=True).rename(col))
    return pd.concat(rebuilt, axis=1)



real_agn_catalog = pd.DataFrame(
    [
        {'Name': 'ASASSN-14ko / ESO 253-G003', 'RAJ2000': 81.3255, 'DEJ2000': -46.0057, 'Type': 'Seyfert/TDE-like repeating nuclear transient', 'z': 0.042489, 'gmag': np.nan},
        {'Name': '3C 273', 'RAJ2000': 187.2779153, 'DEJ2000': 2.0523886, 'Type': 'QSO/blazar', 'z': 0.158339, 'gmag': 12.9},
        {'Name': 'NGC 4151', 'RAJ2000': 182.6357333, 'DEJ2000': 39.4058500, 'Type': 'Seyfert 1.5', 'z': 0.003319, 'gmag': 11.5},
        {'Name': 'NGC 5548', 'RAJ2000': 214.4980583, 'DEJ2000': 25.1367889, 'Type': 'Seyfert 1', 'z': 0.017175, 'gmag': 13.3},
        {'Name': 'Mrk 421', 'RAJ2000': 166.1138080, 'DEJ2000': 38.2088330, 'Type': 'BL Lac/blazar', 'z': 0.030800, 'gmag': 13.0},
        {'Name': 'Mrk 501', 'RAJ2000': 253.4675690, 'DEJ2000': 39.7601690, 'Type': 'BL Lac/blazar', 'z': 0.033660, 'gmag': 13.8},
        {'Name': 'NGC 3516', 'RAJ2000': 166.6978330, 'DEJ2000': 72.5685830, 'Type': 'Seyfert 1.5', 'z': 0.008836, 'gmag': 12.9},
        {'Name': 'NGC 7469', 'RAJ2000': 345.8150750, 'DEJ2000': 8.8739990, 'Type': 'Seyfert 1', 'z': 0.016317, 'gmag': 13.0},
        {'Name': 'NGC 2617', 'RAJ2000': 128.9116250, 'DEJ2000': -4.0882220, 'Type': 'changing-look Seyfert', 'z': 0.014200, 'gmag': 13.2},
        {'Name': 'Ark 120', 'RAJ2000': 79.0475000, 'DEJ2000': -0.1497780, 'Type': 'Seyfert 1', 'z': 0.032713, 'gmag': 13.0},
        {'Name': 'Mrk 335', 'RAJ2000': 1.5813330, 'DEJ2000': 20.2029170, 'Type': 'narrow-line Seyfert 1', 'z': 0.025785, 'gmag': 13.9},
        {'Name': 'Fairall 9', 'RAJ2000': 20.9407500, 'DEJ2000': -58.8057780, 'Type': 'Seyfert 1', 'z': 0.047016, 'gmag': 13.4},
        {'Name': 'NGC 4395', 'RAJ2000': 186.4535960, 'DEJ2000': 33.5469400, 'Type': 'low-mass Seyfert 1', 'z': 0.001064, 'gmag': 11.0},
    ]
)

real_catalog_path = REAL_AGN_DIR / 'real_agn_catalog.csv'
real_agn_catalog.to_csv(real_catalog_path, index=False)

real_targets = build_targets_stage([real_catalog_path], run_dir=REAL_AGN_DIR, source_catalog='curated_real_agn')
print(f'Real AGN targets: {len(real_targets)}')
display(real_targets[['nuclear_target_id', 'source_name', 'ra', 'dec', 'agn_type', 'redshift']])

real_targets_lc = fetch_lightcurves_stage(
    targets=real_targets,
    run_dir=REAL_AGN_DIR,
    backend=REAL_AGN_BACKEND,
    cone_radius_arcsec=REAL_AGN_CONE_RADIUS_ARCSEC,
    n_workers=1,
    verbose=True,
)

# Retry SkyPatrol2 failures with SkyPatrol1. SkyPatrol1 often has historical V-band
# photometry for bright AGN even when SkyPatrol2 returns metadata but no LC block.
failed_fetch = real_targets_lc['fetch_status'].astype(str).eq('failed')
if failed_fetch.any():
    failed_indices = real_targets_lc.index[failed_fetch].to_list()
    fallback_targets = real_targets.loc[failed_fetch.to_numpy()].copy()
    retry_lc = fetch_lightcurves_stage(
        targets=fallback_targets,
        run_dir=REAL_AGN_DIR,
        output=REAL_AGN_DIR / 'parent_agn_targets_with_lightcurves_skypatrol1_retry.parquet',
        cache_dir=REAL_AGN_DIR / 'lightcurves',
        backend=REAL_AGN_FALLBACK_BACKEND,
        cone_radius_arcsec=REAL_AGN_CONE_RADIUS_ARCSEC,
        n_workers=1,
        verbose=True,
    )
    for offset, original_idx in enumerate(failed_indices):
        for col in retry_lc.columns:
            if col not in real_targets_lc.columns:
                real_targets_lc[col] = np.nan
            real_targets_lc.loc[original_idx, col] = retry_lc.iloc[offset][col]
    real_targets_lc = _coerce_real_run_text_columns(real_targets_lc)
    _write_table(real_targets_lc, targets_with_lcs_path(REAL_AGN_DIR))

real_targets_lc = _coerce_real_run_text_columns(real_targets_lc)

fetch_cols = [
    'source_name',
    'fetch_status',
    'asassn_id',
    'cone_sep_arcsec',
    'cone_n_matches',
    'lc_path',
    'fetch_error',
]
fetch_cols = [col for col in fetch_cols if col in real_targets_lc.columns]
print('Fetch status counts after fallback:')
display(real_targets_lc['fetch_status'].value_counts(dropna=False).rename_axis('fetch_status').reset_index(name='n'))
display(real_targets_lc[fetch_cols])

real_features, real_quality = compute_features_stage(
    targets=real_targets_lc,
    run_dir=REAL_AGN_DIR,
    primary_band=REAL_AGN_PRIMARY_BAND,
    min_points=20,
    max_error=0.5,
    include_drw=True,
    verbose=True,
)
real_features['primary_band_used'] = REAL_AGN_PRIMARY_BAND
real_quality['primary_band_used'] = REAL_AGN_PRIMARY_BAND

# Keep g-band primary, but retry V-only rows instead of discarding real AGN with no g-band coverage.
v_retry_mask = (
    real_features['nuc_feature_status'].astype(str).eq('failed_quality')
    & pd.to_numeric(real_quality.get('n_v_points'), errors='coerce').fillna(0).ge(20)
)
if v_retry_mask.any():
    retry_indices = real_features.index[v_retry_mask].to_list()
    v_features, v_quality = compute_features_stage(
        targets=real_targets_lc.loc[retry_indices].copy(),
        run_dir=REAL_AGN_DIR,
        features_output=REAL_AGN_DIR / 'stochastic_features_v_retry.parquet',
        quality_output=REAL_AGN_DIR / 'lightcurve_quality_v_retry.parquet',
        primary_band=REAL_AGN_FALLBACK_BAND,
        min_points=20,
        max_error=0.5,
        include_drw=True,
        verbose=True,
    )
    v_features['primary_band_used'] = REAL_AGN_FALLBACK_BAND
    v_quality['primary_band_used'] = REAL_AGN_FALLBACK_BAND
    for offset, original_idx in enumerate(retry_indices):
        for col in v_features.columns:
            if col not in real_features.columns:
                real_features[col] = np.nan
            real_features.loc[original_idx, col] = v_features.iloc[offset][col]
        for col in v_quality.columns:
            if col not in real_quality.columns:
                real_quality[col] = np.nan
            real_quality.loc[original_idx, col] = v_quality.iloc[offset][col]
    real_features = _coerce_real_run_text_columns(real_features)
    real_quality = _coerce_real_run_text_columns(real_quality)
    _write_table(real_features, features_path(REAL_AGN_DIR))
    _write_table(real_quality, quality_path(REAL_AGN_DIR))

real_features = _coerce_real_run_text_columns(real_features)
real_quality = _coerce_real_run_text_columns(real_quality)

print('Feature status counts:')
display(real_features['nuc_feature_status'].value_counts(dropna=False).rename_axis('nuc_feature_status').reset_index(name='n'))
print('Quality status counts:')
display(real_quality['quality_status'].value_counts(dropna=False).rename_axis('quality_status').reset_index(name='n'))
print('Primary band used:')
display(real_features['primary_band_used'].value_counts(dropna=False).rename_axis('primary_band_used').reset_index(name='n'))

real_ranked = rank_outliers_stage(features=real_features, run_dir=REAL_AGN_DIR)
real_review = export_review_stage(ranked=real_ranked, run_dir=REAL_AGN_DIR, min_score=0.0)

real_rank_cols = [
    'nuc_rank',
    'source_name',
    'primary_band_used',
    'nuc_outlier_label',
    'nuc_outlier_score',
    'nuc_state_change_score',
    'nuc_transient_like_score',
    'nuc_excess_variance',
    'nuc_fractional_rms',
    'nuc_drw_tau_days',
    'nuc_drw_reliable',
    'nuc_stoch_reliability_score',
    'nuc_feature_status',
]
real_rank_cols = [col for col in real_rank_cols if col in real_ranked.columns]

display(real_ranked[real_rank_cols].head(25))
display(real_review[['candidate_id', 'event_class', 'interest_score', 'morphology_primary', 'review_notes']].head(25))

print(f'Wrote real AGN outputs to: {REAL_AGN_DIR}')


Real AGN targets: 13


,nuclear_target_id,source_name,ra,dec,agn_type,redshift
0,curated_real_agn:3C_273,3C 273,187.277915,2.052389,QSO/blazar,0.158339
1,curated_real_agn:ASASSN-14ko_ESO_253-G003,ASASSN-14ko / ESO 253-G003,81.325500,-46.005700,Seyfert/TDE-like repeating nuclear transient,0.042489
2,curated_real_agn:Ark_120,Ark 120,79.047500,-0.149778,Seyfert 1,0.032713
3,curated_real_agn:Fairall_9,Fairall 9,20.940750,-58.805778,Seyfert 1,0.047016
4,curated_real_agn:Mrk_335,Mrk 335,1.581333,20.202917,narrow-line Seyfert 1,0.025785
5,curated_real_agn:Mrk_421,Mrk 421,166.113808,38.208833,BL Lac/blazar,0.030800
6,curated_real_agn:Mrk_501,Mrk 501,253.467569,39.760169,BL Lac/blazar,0.033660
7,curated_real_agn:NGC_2617,NGC 2617,128.911625,-4.088222,changing-look Seyfert,0.014200
8,curated_real_agn:NGC_3516,NGC 3516,166.697833,72.568583,Seyfert 1.5,0.008836
9,curated_real_agn:NGC_4151,NGC 4151,182.635733,39.405850,Seyfert 1.5,0.003319



nuclear-fetch:   0%|          | 0/13 [00:00<?, ?it/s]


nuclear-fetch:   8%|▊         | 1/13 [00:01<00:19,  1.63s/it]


nuclear-fetch:  15%|█▌        | 2/13 [00:04<00:26,  2.45s/it]

...

Primary band used:


,primary_band_used,n
0,g,11
1,v,2


,nuc_rank,source_name,primary_band_used,nuc_outlier_label,nuc_outlier_score,nuc_state_change_score,nuc_transient_like_score,nuc_excess_variance,nuc_fractional_rms,nuc_drw_tau_days,nuc_drw_reliable,nuc_stoch_reliability_score,nuc_feature_status
0,1,Mrk 421,g,state_change_candidate,14.333475,5.0,0.810176,0.032763,0.181006,12.273458,True,1.00,ok
1,2,NGC 4151,g,state_change_candidate,12.870276,5.0,0.603768,0.050799,0.225386,NaN,False,0.65,ok
2,3,Mrk 335,g,state_change_candidate,10.808501,5.0,0.552774,0.013545,0.116381,17.610893,True,1.00,ok
3,4,Ark 120,g,state_change_candidate,10.403226,5.0,0.445707,0.009976,0.099881,13.564989,True,1.00,ok
4,5,NGC 7469,g,state_change_candidate,9.224023,5.0,0.332781,0.008347,0.091363,6.857568,True,1.00,ok
5,6,3C 273,g,state_change_candidate,8.869626,5.0,1.262103,0.008911,0.094400,NaN,False,0.65,ok
6,7,NGC 4395,g,state_change_candidate,8.395083,5.0,0.978631,0.001814,0.042586,NaN,False,0.65,ok
7,8,NGC 5548,g,state_change_candidate,8.381176,5.0,0.528182,0.008746,0.093519,5.704254,True,1.00,ok
8,9,Fairall 9,v,state_change_candidate,8.345133,5.0,0.644757,0.003973,0.063028,28.372724,True,1.00,ok
9,10,ASASSN-14ko / ESO 253-G003,v,state_change_candidate,8.325682,5.0,0.804059,0.001237,0.035165,6.835205,True,1.00,ok


,candidate_id,event_class,interest_score,morphology_primary,review_notes
0,curated_real_agn:Mrk_421,nuclear_agn_outlier,14.333475,state_change_candidate,AGN stochastic outlier: state_change_candidate...
1,curated_real_agn:NGC_4151,nuclear_agn_outlier,12.870276,state_change_candidate,AGN stochastic outlier: state_change_candidate...
2,curated_real_agn:Mrk_335,nuclear_agn_outlier,10.808501,state_change_candidate,AGN stochastic outlier: state_change_candidate...
3,curated_real_agn:Ark_120,nuclear_agn_outlier,10.403226,state_change_candidate,AGN stochastic outlier: state_change_candidate...
4,curated_real_agn:NGC_7469,nuclear_agn_outlier,9.224023,state_change_candidate,AGN stochastic outlier: state_change_candidate...
5,curated_real_agn:3C_273,nuclear_agn_outlier,8.869626,state_change_candidate,AGN stochastic outlier: state_change_candidate...
6,curated_real_agn:NGC_4395,nuclear_agn_outlier,8.395083,state_change_candidate,AGN stochastic outlier: state_change_candidate...
7,curated_real_agn:NGC_5548,nuclear_agn_outlier,8.381176,state_change_candidate,AGN stochastic outlier: state_change_candidate...
8,curated_real_agn:Fairall_9,nuclear_agn_outlier,8.345133,state_change_candidate,AGN stochastic outlier: state_change_candidate...
9,curated_real_agn:ASASSN-14ko_ESO_253-G003,nuclear_agn_outlier,8.325682,state_change_candidate,AGN stochastic outlier: state_change_candidate...


Wrote real AGN outputs to: /Users/calder/code/malca/output/runs/agn_stochastic_real_agn


## Literature AGN / MilliQuas ASAS-SN Batch

This section uses the native SkyPatrol `milliquas` input catalog as the all-literature AGN parent sample. MilliQuas v8 is the broad literature compendium of published QSOs/AGN, and SkyPatrol's `milliquas` table already includes `asas_sn_id` values, so this is much faster than cone-searching every literature coordinate one at a time.

The defaults below compile the full SkyPatrol-native MilliQuas parent catalog only. The index is checkpointed under `output/runs/agn_stochastic_milliquas/milliquas_index_chunks/`, so the full compile can be interrupted and resumed. Light-curve downloads and stochastic feature batches are disabled by default; set `MILLIQUAS_RUN_FEATURE_BATCHES = True` only when you intentionally want to launch that slower stage.


In [13]:
# Literature AGN catalog: SkyPatrol-native MilliQuas parent sample.
# Defaults compile the full catalog only; feature batches stay opt-in.

from __future__ import annotations

import math
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from pyasassn.client import SkyPatrolClient
from tqdm.auto import tqdm

MILLIQUAS_RUN_DIR = REPO_ROOT / 'output' / 'runs' / 'agn_stochastic_milliquas'
MILLIQUAS_RUN_DIR.mkdir(parents=True, exist_ok=True)

MILLIQUAS_INDEX_CHUNK_SIZE = 10_000
MILLIQUAS_INDEX_MAX_ROWS = None        # None compiles all SkyPatrol MilliQuas rows.
MILLIQUAS_COMPILE_CATALOG = True
MILLIQUAS_RUN_FEATURE_BATCHES = False  # Leave False unless you want LC downloads/features.
MILLIQUAS_USE_LOCAL_CLUSTER_LCS = True
MILLIQUAS_REQUIRE_LOCAL_LC_MATCH_FOR_FEATURES = True
MILLIQUAS_LOCAL_LC_EXT_PRIORITY = ('dat3', 'dat2', 'dat', 'csv')
MILLIQUAS_LOCAL_LC_DIRS = [
    *[Path(path).expanduser() for path in os.environ.get('MALCA_CLUSTER_LC_DIRS', '').split(os.pathsep) if path.strip()],
    REPO_ROOT / 'output' / 'runs' / 'runs_march18_bundle_all' / 'bundle_assets' / 'lightcurves',
    REPO_ROOT / 'output' / 'runs' / 'ltv_march18' / 'bundle_assets' / 'lightcurves',
]

# Feature-stage safety caps. These are ignored unless MILLIQUAS_RUN_FEATURE_BATCHES is True.
MILLIQUAS_FEATURE_MAX_TARGETS = 25
MILLIQUAS_BATCH_SIZE = 25
MILLIQUAS_MAX_BATCHES_THIS_RUN = 1     # Set None to run every feature batch.
MILLIQUAS_PRIMARY_BAND = 'g'
MILLIQUAS_FALLBACK_BAND = 'v'
MILLIQUAS_BACKEND = 'skypatrol2'

MILLIQUAS_INDEX_FILENAME = 'skypatrol_milliquas_asassn_index.parquet'
MILLIQUAS_TARGETS_FILENAME = 'milliquas_parent_agn_targets.parquet'
MILLIQUAS_LOCAL_LC_MANIFEST_FILENAME = 'local_cluster_lightcurve_manifest.parquet'
MILLIQUAS_TARGETS_WITH_LOCAL_LCS_FILENAME = 'milliquas_parent_agn_targets_with_local_lightcurves.parquet'


def _coerce_parquet_safe_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce mixed/object columns to strings before parquet writes."""
    rebuilt = []
    for pos, col in enumerate(df.columns):
        series = df.iloc[:, pos]
        if series.dtype == object or str(series.dtype).startswith('string'):
            series = series.map(lambda value: '' if pd.isna(value) else str(value)).astype('string')
        rebuilt.append(series.reset_index(drop=True).rename(col))
    return pd.concat(rebuilt, axis=1)


def skypatrol_count_catalog_rows(catalog: str = 'milliquas') -> int:
    client = SkyPatrolClient(verbose=False)
    count = client.adql_query(f'SELECT COUNT(*) AS n FROM {catalog}', download=False)
    return int(count.iloc[0]['n'])


def _normalize_milliquas_index_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Return a sorted, unique SkyPatrol MilliQuas index frame."""
    cols = ['asas_sn_id', 'ra_deg', 'dec_deg', 'name', 'broad_type', 'rmag', 'bmag', 'redshift', 'qso_prob', 'source_catalog', 'has_asassn_source_id']
    if df is None or df.empty:
        return pd.DataFrame(columns=cols)
    if 'asas_sn_id' not in df.columns:
        raise ValueError('SkyPatrol MilliQuas index is missing asas_sn_id')

    out = df.copy()
    out['asas_sn_id'] = pd.to_numeric(out['asas_sn_id'], errors='coerce').astype('Int64')
    out = out.dropna(subset=['asas_sn_id']).drop_duplicates('asas_sn_id', keep='first')
    out = out.sort_values('asas_sn_id').reset_index(drop=True)
    return out


def fetch_skypatrol_milliquas_index(
    *,
    run_dir: Path = MILLIQUAS_RUN_DIR,
    chunk_size: int = MILLIQUAS_INDEX_CHUNK_SIZE,
    max_rows: int | None = MILLIQUAS_INDEX_MAX_ROWS,
    total_available: int | None = None,
    force: bool = False,
) -> pd.DataFrame:
    """Build/resume the SkyPatrol MilliQuas ASAS-SN source index."""
    run_dir = Path(run_dir)
    chunk_dir = run_dir / 'milliquas_index_chunks'
    chunk_dir.mkdir(parents=True, exist_ok=True)
    final_path = run_dir / MILLIQUAS_INDEX_FILENAME
    if int(chunk_size) <= 0:
        raise ValueError('chunk_size must be positive')

    requested_rows = None if max_rows is None else int(max_rows)
    if requested_rows is not None and requested_rows < 0:
        raise ValueError('max_rows must be non-negative or None')
    if max_rows is None:
        total_available = int(total_available) if total_available is not None else skypatrol_count_catalog_rows('milliquas')
        target_rows = total_available
    else:
        target_rows = requested_rows

    if final_path.exists() and not force:
        cached = _normalize_milliquas_index_frame(pd.read_parquet(final_path))
        if len(cached) >= int(target_rows):
            return cached.head(int(target_rows)).copy()

    existing_chunks = sorted(chunk_dir.glob('milliquas_index_*.parquet'))
    chunk_frames = [pd.read_parquet(path) for path in existing_chunks]
    if chunk_frames:
        current = _normalize_milliquas_index_frame(pd.concat(chunk_frames, ignore_index=True))
        last_id = int(pd.to_numeric(current['asas_sn_id'], errors='coerce').max())
    else:
        current = _normalize_milliquas_index_frame(pd.DataFrame())
        last_id = 0

    if max_rows is not None:
        total_available = int(total_available) if total_available is not None else skypatrol_count_catalog_rows('milliquas')
        target_rows = min(int(target_rows), total_available)
    if len(current) >= int(target_rows):
        current = current.head(int(target_rows)).copy()
        current = _coerce_parquet_safe_columns(current)
        current.to_parquet(final_path, index=False)
        return current

    client = SkyPatrolClient(verbose=False)
    cols = 'asas_sn_id, ra_deg, dec_deg, name, broad_type, rmag, bmag, redshift, qso_prob'
    next_chunk_idx = len(existing_chunks)

    pbar = tqdm(total=target_rows, initial=min(len(current), target_rows), desc='milliquas-index')
    while len(current) < target_rows:
        remaining = target_rows - len(current)
        limit = min(int(chunk_size), int(remaining))
        before_len = len(current)
        query = (
            f'SELECT {cols} FROM milliquas '
            f'WHERE asas_sn_id > {last_id} '
            f'ORDER BY asas_sn_id LIMIT {limit}'
        )
        chunk = client.adql_query(query, download=False)
        if chunk is None or chunk.empty:
            break
        chunk = chunk.copy()
        chunk['source_catalog'] = 'skypatrol_milliquas'
        chunk['has_asassn_source_id'] = True
        chunk = _coerce_parquet_safe_columns(chunk)
        chunk_path = chunk_dir / f'milliquas_index_{next_chunk_idx:06d}_{last_id}.parquet'
        chunk.to_parquet(chunk_path, index=False)
        next_chunk_idx += 1
        if current.empty:
            current = _normalize_milliquas_index_frame(chunk)
        else:
            current = _normalize_milliquas_index_frame(pd.concat([current, chunk], ignore_index=True))
        last_id = int(current['asas_sn_id'].max())
        pbar.update(min(len(current), target_rows) - min(before_len, target_rows))
        if len(chunk) < limit or len(current) <= before_len:
            break
    pbar.close()

    current = current.head(target_rows).copy()
    current = _coerce_parquet_safe_columns(current)
    current.to_parquet(final_path, index=False)
    return current


def milliquas_targets_from_index(index: pd.DataFrame, *, run_dir: Path = MILLIQUAS_RUN_DIR) -> pd.DataFrame:
    """Normalize SkyPatrol MilliQuas rows into the notebook AGN target schema."""
    source = index.copy()
    source['Name'] = source['name']
    source['RAJ2000'] = pd.to_numeric(source['ra_deg'], errors='coerce')
    source['DEJ2000'] = pd.to_numeric(source['dec_deg'], errors='coerce')
    source['Type'] = source['broad_type'].astype(str)
    source['z'] = pd.to_numeric(source['redshift'], errors='coerce')
    source['rmag'] = pd.to_numeric(source['rmag'], errors='coerce')
    source['bmag'] = pd.to_numeric(source['bmag'], errors='coerce')
    source['asassn_id'] = source['asas_sn_id'].astype(str)

    targets = normalize_agn_catalog(source, source_catalog='skypatrol_milliquas')
    extra_cols = [col for col in ['asassn_id', 'qso_prob', 'broad_type'] if col in source.columns]
    if extra_cols and not targets.empty:
        extras = source[extra_cols].copy()
        extras['asassn_id'] = extras['asassn_id'].astype(str)
        extras = extras.drop_duplicates('asassn_id', keep='first')
        merge_cols = [col for col in extras.columns if col != 'asassn_id']
        targets = targets.drop(columns=[col for col in merge_cols if col in targets.columns])
        targets = targets.merge(extras, on='asassn_id', how='left')
    targets = _coerce_parquet_safe_columns(targets)
    out_path = Path(run_dir) / MILLIQUAS_TARGETS_FILENAME
    targets.to_parquet(out_path, index=False)
    return targets


def summarize_milliquas_catalog(
    index: pd.DataFrame,
    targets: pd.DataFrame,
    *,
    total_rows: int | None = None,
) -> pd.DataFrame:
    """Return lightweight QA counts for a compiled MilliQuas catalog."""
    if targets is None or targets.empty:
        valid_coord = 0
        with_asassn_id = 0
        unique_target_ids = 0
    else:
        ra = pd.to_numeric(targets.get('ra'), errors='coerce')
        dec = pd.to_numeric(targets.get('dec'), errors='coerce')
        valid_coord = int((ra.between(0.0, 360.0, inclusive='left') & dec.between(-90.0, 90.0)).sum())
        with_asassn_id = int(targets.get('asassn_id', pd.Series(dtype=object)).astype(str).str.strip().ne('').sum())
        unique_target_ids = int(targets.get('nuclear_target_id', pd.Series(dtype=object)).nunique())

    return pd.DataFrame([
        {
            'sky_patrol_rows_reported': np.nan if total_rows is None else int(total_rows),
            'indexed_rows': int(len(index)) if index is not None else 0,
            'normalized_targets': int(len(targets)) if targets is not None else 0,
            'unique_target_ids': unique_target_ids,
            'valid_coordinates': valid_coord,
            'with_asassn_id': with_asassn_id,
        }
    ])


def _show_table(df: pd.DataFrame) -> None:
    """Display in notebooks, print in plain Python."""
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def _asassn_id_from_lightcurve_path(path: Path) -> str:
    """Extract the ASAS-SN source ID from a local light-curve filename."""
    match = re.match(r'^(\d+)', path.stem)
    return match.group(1) if match else ''


def build_local_lightcurve_manifest(
    lc_dirs: list[str | Path] | tuple[str | Path, ...] = tuple(MILLIQUAS_LOCAL_LC_DIRS),
    *,
    run_dir: Path = MILLIQUAS_RUN_DIR,
    suffix_priority: tuple[str, ...] = MILLIQUAS_LOCAL_LC_EXT_PRIORITY,
    output: str | Path | None = None,
) -> pd.DataFrame:
    """Build a manifest mapping ASAS-SN IDs to existing local cluster light curves."""
    run_dir = Path(run_dir)
    out_path = Path(output) if output is not None else run_dir / MILLIQUAS_LOCAL_LC_MANIFEST_FILENAME
    allowed = {str(ext).lower().lstrip('.') for ext in suffix_priority}
    priority = {ext: i for i, ext in enumerate(suffix_priority)}

    rows: list[dict[str, object]] = []
    seen_dirs: set[Path] = set()
    for dir_order, raw_dir in enumerate(lc_dirs):
        lc_dir = Path(raw_dir).expanduser()
        if not lc_dir.is_dir():
            continue
        resolved_dir = lc_dir.resolve()
        if resolved_dir in seen_dirs:
            continue
        seen_dirs.add(resolved_dir)

        for path in lc_dir.rglob('*'):
            if not path.is_file():
                continue
            suffix = path.suffix.lower().lstrip('.')
            if suffix not in allowed:
                continue
            asas_sn_id = _asassn_id_from_lightcurve_path(path)
            if not asas_sn_id:
                continue
            stat = path.stat()
            rows.append(
                {
                    'asas_sn_id': asas_sn_id,
                    'lc_path': str(path.resolve()),
                    'lc_filename': path.name,
                    'lc_suffix': suffix,
                    'lc_dir': str(resolved_dir),
                    'lc_size_bytes': int(stat.st_size),
                    'lc_mtime': float(stat.st_mtime),
                    'lc_suffix_priority': int(priority[suffix]),
                    'lc_dir_order': int(dir_order),
                }
            )

    manifest = pd.DataFrame(rows)
    if manifest.empty:
        manifest = pd.DataFrame(
            columns=[
                'asas_sn_id',
                'lc_path',
                'lc_filename',
                'lc_suffix',
                'lc_dir',
                'lc_size_bytes',
                'lc_mtime',
                'lc_suffix_priority',
                'lc_dir_order',
            ]
        )
    else:
        manifest = manifest.sort_values(['asas_sn_id', 'lc_suffix_priority', 'lc_dir_order', 'lc_path'])
        manifest = manifest.drop_duplicates('asas_sn_id', keep='first').reset_index(drop=True)
        manifest = _coerce_parquet_safe_columns(manifest)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_parquet(out_path, index=False)
    return manifest


def attach_local_lightcurve_paths(
    targets: pd.DataFrame,
    manifest: pd.DataFrame,
    *,
    run_dir: Path = MILLIQUAS_RUN_DIR,
    output: str | Path | None = None,
) -> pd.DataFrame:
    """Attach existing local cluster light-curve paths to normalized MilliQuas targets."""
    run_dir = Path(run_dir)
    out_path = Path(output) if output is not None else run_dir / MILLIQUAS_TARGETS_WITH_LOCAL_LCS_FILENAME
    work = targets.copy()
    if work.empty:
        work['local_lc_status'] = pd.Series(dtype='string')
        out_path.parent.mkdir(parents=True, exist_ok=True)
        work.to_parquet(out_path, index=False)
        return work

    work['asassn_id'] = work['asassn_id'].astype(str).str.strip()
    local = manifest.copy()
    if local.empty:
        local = pd.DataFrame(columns=['asas_sn_id', 'lc_path', 'lc_suffix', 'lc_dir', 'lc_filename'])
    local['asas_sn_id'] = local['asas_sn_id'].astype(str).str.strip()
    local = local.rename(
        columns={
            'lc_path': 'local_lc_path',
            'lc_suffix': 'local_lc_suffix',
            'lc_dir': 'local_lc_dir',
            'lc_filename': 'local_lc_filename',
        }
    )
    local_cols = ['asas_sn_id', 'local_lc_path', 'local_lc_suffix', 'local_lc_dir', 'local_lc_filename']
    local_cols = [col for col in local_cols if col in local.columns]
    out = work.merge(local[local_cols], left_on='asassn_id', right_on='asas_sn_id', how='left')
    out = out.drop(columns=['asas_sn_id'], errors='ignore')
    out['local_lc_status'] = np.where(out['local_lc_path'].fillna('').astype(str).str.strip().ne(''), 'matched_local_lc', 'missing_local_lc')
    out['catalog_lc_path'] = out['lc_path'].astype(str) if 'lc_path' in out.columns else ''
    out['lc_path'] = out['local_lc_path'].fillna('').astype(str)
    out = _coerce_parquet_safe_columns(out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(out_path, index=False)
    return out


def summarize_local_lightcurve_matches(targets_with_lcs: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    """Return QA counts for the local cluster LC match layer."""
    matched = 0
    existing = 0
    if targets_with_lcs is not None and not targets_with_lcs.empty:
        matched_mask = targets_with_lcs.get('local_lc_status', pd.Series(dtype=object)).astype(str).eq('matched_local_lc')
        matched = int(matched_mask.sum())
        paths = targets_with_lcs.loc[matched_mask, 'lc_path'].astype(str) if 'lc_path' in targets_with_lcs.columns else pd.Series(dtype=object)
        existing = int(paths.map(lambda value: Path(value).is_file()).sum())

    return pd.DataFrame(
        [
            {
                'manifest_lightcurves': int(len(manifest)) if manifest is not None else 0,
                'targets_total': int(len(targets_with_lcs)) if targets_with_lcs is not None else 0,
                'targets_with_local_lc': matched,
                'matched_lc_paths_existing': existing,
            }
        ]
    )


def compile_milliquas_catalog(
    *,
    run_dir: Path = MILLIQUAS_RUN_DIR,
    chunk_size: int = MILLIQUAS_INDEX_CHUNK_SIZE,
    max_rows: int | None = MILLIQUAS_INDEX_MAX_ROWS,
    force: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Compile the SkyPatrol MilliQuas index and normalized AGN target catalog."""
    total_rows = skypatrol_count_catalog_rows('milliquas')
    print(f'SkyPatrol milliquas rows with ASAS-SN source IDs: {total_rows:,}')

    index = fetch_skypatrol_milliquas_index(
        run_dir=run_dir,
        chunk_size=chunk_size,
        max_rows=max_rows,
        total_available=total_rows,
        force=force,
    )
    print(f'Indexed rows in this catalog compile: {len(index):,}')
    _show_table(index.head())

    targets = milliquas_targets_from_index(index, run_dir=run_dir)
    print(f'Normalized MilliQuas targets: {len(targets):,}')
    _show_table(targets.head())

    summary = summarize_milliquas_catalog(index, targets, total_rows=total_rows)
    print('MilliQuas catalog summary:')
    _show_table(summary)
    if 'source_catalog' in targets.columns:
        print('Source catalog counts:')
        _show_table(targets['source_catalog'].value_counts(dropna=False).rename_axis('source_catalog').reset_index(name='n'))
    return index, targets, summary


def _retry_feature_rows_with_v_band(
    features: pd.DataFrame,
    quality: pd.DataFrame,
    targets_lc: pd.DataFrame,
    *,
    run_dir: Path,
    primary_band: str = MILLIQUAS_FALLBACK_BAND,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Retry V-band for rows that have no usable g but enough V points."""
    if features.empty or quality.empty:
        return features, quality
    retry_mask = (
        features['nuc_feature_status'].astype(str).eq('failed_quality')
        & pd.to_numeric(quality.get('n_v_points'), errors='coerce').fillna(0).ge(20)
    )
    if not retry_mask.any():
        return features, quality

    retry_indices = features.index[retry_mask].to_list()
    v_features, v_quality = compute_features_stage(
        targets=targets_lc.loc[retry_indices].copy(),
        run_dir=run_dir,
        features_output=Path(run_dir) / 'stochastic_features_v_retry_latest.parquet',
        quality_output=Path(run_dir) / 'lightcurve_quality_v_retry_latest.parquet',
        primary_band=primary_band,
        min_points=20,
        max_error=0.5,
        include_drw=True,
        verbose=False,
    )
    v_features['primary_band_used'] = primary_band
    v_quality['primary_band_used'] = primary_band

    for offset, original_idx in enumerate(retry_indices):
        for col in v_features.columns:
            if col not in features.columns:
                features[col] = np.nan
            features.loc[original_idx, col] = v_features.iloc[offset][col]
        for col in v_quality.columns:
            if col not in quality.columns:
                quality[col] = np.nan
            quality.loc[original_idx, col] = v_quality.iloc[offset][col]
    return features, quality


def run_milliquas_lightcurve_feature_batches(
    targets: pd.DataFrame,
    *,
    run_dir: Path = MILLIQUAS_RUN_DIR,
    max_targets: int | None = MILLIQUAS_FEATURE_MAX_TARGETS,
    batch_size: int = MILLIQUAS_BATCH_SIZE,
    max_batches_this_run: int | None = MILLIQUAS_MAX_BATCHES_THIS_RUN,
    backend: str = MILLIQUAS_BACKEND,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Download and compute AGN features in resumable batches."""
    run_dir = Path(run_dir)
    batch_dir = run_dir / 'feature_batches'
    batch_dir.mkdir(parents=True, exist_ok=True)
    work = targets.copy()
    if max_targets is not None:
        work = work.head(int(max_targets)).copy()

    feature_frames, quality_frames = [], []
    batches_started = 0
    for start in range(0, len(work), int(batch_size)):
        stop = min(start + int(batch_size), len(work))
        tag = f'{start:08d}_{stop:08d}'
        features_out = batch_dir / f'stochastic_features_{tag}.parquet'
        quality_out = batch_dir / f'lightcurve_quality_{tag}.parquet'
        targets_lc_out = batch_dir / f'targets_with_lightcurves_{tag}.parquet'

        if features_out.exists() and quality_out.exists():
            feature_frames.append(pd.read_parquet(features_out))
            quality_frames.append(pd.read_parquet(quality_out))
            continue
        if max_batches_this_run is not None and batches_started >= int(max_batches_this_run):
            continue

        batch_targets = work.iloc[start:stop].copy()
        batch_targets_lc = fetch_lightcurves_stage(
            targets=batch_targets,
            run_dir=run_dir,
            output=targets_lc_out,
            cache_dir=run_dir / 'lightcurves',
            backend=backend,
            n_workers=1,
            verbose=True,
        )
        batch_features, batch_quality = compute_features_stage(
            targets=batch_targets_lc,
            run_dir=run_dir,
            features_output=features_out,
            quality_output=quality_out,
            primary_band=MILLIQUAS_PRIMARY_BAND,
            min_points=20,
            max_error=0.5,
            include_drw=True,
            verbose=True,
        )
        batch_features['primary_band_used'] = MILLIQUAS_PRIMARY_BAND
        batch_quality['primary_band_used'] = MILLIQUAS_PRIMARY_BAND
        batch_features, batch_quality = _retry_feature_rows_with_v_band(
            batch_features,
            batch_quality,
            batch_targets_lc,
            run_dir=run_dir,
            primary_band=MILLIQUAS_FALLBACK_BAND,
        )
        batch_features = _coerce_parquet_safe_columns(batch_features)
        batch_quality = _coerce_parquet_safe_columns(batch_quality)
        batch_features.to_parquet(features_out, index=False)
        batch_quality.to_parquet(quality_out, index=False)
        feature_frames.append(batch_features)
        quality_frames.append(batch_quality)
        batches_started += 1

    features = pd.concat(feature_frames, ignore_index=True) if feature_frames else pd.DataFrame()
    quality = pd.concat(quality_frames, ignore_index=True) if quality_frames else pd.DataFrame()
    if not features.empty:
        features = _coerce_parquet_safe_columns(features)
        quality = _coerce_parquet_safe_columns(quality)
        features.to_parquet(features_path(run_dir), index=False)
        quality.to_parquet(quality_path(run_dir), index=False)
        ranked = rank_outliers_stage(features=features, run_dir=run_dir)
        review = export_review_stage(ranked=ranked, run_dir=run_dir, min_score=0.0)
    else:
        ranked = pd.DataFrame()
        review = pd.DataFrame()
    return features, quality, ranked, review


if MILLIQUAS_COMPILE_CATALOG:
    milliquas_index, milliquas_targets, milliquas_summary = compile_milliquas_catalog(
        run_dir=MILLIQUAS_RUN_DIR,
        chunk_size=MILLIQUAS_INDEX_CHUNK_SIZE,
        max_rows=MILLIQUAS_INDEX_MAX_ROWS,
    )
else:
    milliquas_index = pd.DataFrame()
    milliquas_targets_path = MILLIQUAS_RUN_DIR / MILLIQUAS_TARGETS_FILENAME
    milliquas_targets = pd.read_parquet(milliquas_targets_path) if milliquas_targets_path.exists() else pd.DataFrame()
    milliquas_summary = summarize_milliquas_catalog(milliquas_index, milliquas_targets)
    print('Skipped MilliQuas catalog compile; using cached targets if present.')

if MILLIQUAS_USE_LOCAL_CLUSTER_LCS:
    milliquas_local_lc_manifest = build_local_lightcurve_manifest(
        MILLIQUAS_LOCAL_LC_DIRS,
        run_dir=MILLIQUAS_RUN_DIR,
    )
    print(f'Local cluster light curves indexed: {len(milliquas_local_lc_manifest):,}')
    _show_table(milliquas_local_lc_manifest.head())

    milliquas_targets_with_lcs = attach_local_lightcurve_paths(
        milliquas_targets,
        milliquas_local_lc_manifest,
        run_dir=MILLIQUAS_RUN_DIR,
    )
    milliquas_local_lc_summary = summarize_local_lightcurve_matches(milliquas_targets_with_lcs, milliquas_local_lc_manifest)
    print('Local cluster LC match summary:')
    _show_table(milliquas_local_lc_summary)
else:
    milliquas_local_lc_manifest = pd.DataFrame()
    milliquas_targets_with_lcs = milliquas_targets.copy()
    milliquas_local_lc_summary = summarize_local_lightcurve_matches(milliquas_targets_with_lcs, milliquas_local_lc_manifest)
    print('Skipped local cluster LC manifest/matching.')

if MILLIQUAS_REQUIRE_LOCAL_LC_MATCH_FOR_FEATURES:
    status_series = milliquas_targets_with_lcs.get('local_lc_status', pd.Series('', index=milliquas_targets_with_lcs.index))
    path_series = milliquas_targets_with_lcs.get('lc_path', pd.Series('', index=milliquas_targets_with_lcs.index))
    matched_local_lc = status_series.astype(str).eq('matched_local_lc')
    existing_local_lc = path_series.astype(str).map(lambda value: Path(value).is_file())
    milliquas_analysis_targets = milliquas_targets_with_lcs.loc[matched_local_lc & existing_local_lc].copy()
else:
    milliquas_analysis_targets = milliquas_targets_with_lcs.copy()
print(f'MilliQuas analysis targets with local LC paths: {len(milliquas_analysis_targets):,}')

if MILLIQUAS_RUN_FEATURE_BATCHES and not milliquas_analysis_targets.empty:
    milliquas_features, milliquas_quality, milliquas_ranked, milliquas_review = run_milliquas_lightcurve_feature_batches(
        milliquas_analysis_targets,
        run_dir=MILLIQUAS_RUN_DIR,
        max_targets=MILLIQUAS_FEATURE_MAX_TARGETS,
        batch_size=MILLIQUAS_BATCH_SIZE,
        max_batches_this_run=MILLIQUAS_MAX_BATCHES_THIS_RUN,
        backend=MILLIQUAS_BACKEND,
    )

    if not milliquas_features.empty:
        print('MilliQuas feature status counts:')
        _show_table(milliquas_features['nuc_feature_status'].value_counts(dropna=False).rename_axis('nuc_feature_status').reset_index(name='n'))
        print('MilliQuas primary band used:')
        _show_table(milliquas_features['primary_band_used'].value_counts(dropna=False).rename_axis('primary_band_used').reset_index(name='n'))
        _show_table(milliquas_ranked[[
            'nuc_rank',
            'source_name',
            'primary_band_used',
            'nuc_outlier_label',
            'nuc_outlier_score',
            'nuc_n_points',
            'nuc_median_mag',
            'nuc_feature_status',
        ]].head(25))
    else:
        print('No new MilliQuas feature batches were computed in this run.')
else:
    milliquas_features = pd.DataFrame()
    milliquas_quality = pd.DataFrame()
    milliquas_ranked = pd.DataFrame()
    milliquas_review = pd.DataFrame()
    print('Skipped MilliQuas feature batches; set MILLIQUAS_RUN_FEATURE_BATCHES = True to run them.')

print(f'Wrote/resumed MilliQuas outputs under: {MILLIQUAS_RUN_DIR}')


SkyPatrol milliquas rows with ASAS-SN source IDs: 1,979,676
Indexed rows in this notebook run: 10,000


,asas_sn_id,ra_deg,dec_deg,name,broad_type,rmag,bmag,redshift,qso_prob,source_catalog,has_asassn_source_id
0,3539,96.150768,-79.506215,WISEA J062436.17-793022.3,q,18.73,19.48,2.3,98.0,skypatrol_milliquas,True
1,14515,108.215905,-79.101620,WISEA J071251.81-790605.8,q,17.33,17.92,1.0,98.0,skypatrol_milliquas,True
2,20868,63.932729,-79.434532,WISEA J041543.84-792604.3,q,18.00,18.65,1.2,100.0,skypatrol_milliquas,True
3,21518,38.462832,-79.650275,WISEA J023351.10-793900.9,q,18.29,18.80,2.3,100.0,skypatrol_milliquas,True
4,28413,25.058815,-79.879270,WISEA J014014.10-795245.4,q,17.86,18.32,0.6,100.0,skypatrol_milliquas,True


Normalized MilliQuas targets: 10,000


,nuclear_target_id,source_catalog,source_name,ra,dec,agn_type,redshift,asassn_id,gaia_id,g_mag,...,r_mag,i_mag,wise_w1,wise_w2,black_hole_mass,bol_luminosity,eddington_ratio,lc_path,qso_prob,broad_type
0,skypatrol_milliquas:1AXG_J170305_4526,skypatrol_milliquas,1AXG J170305+4526,255.776968,45.432574,QX,0.171,51539868664,,NaN,...,16.96,NaN,NaN,NaN,NaN,NaN,NaN,,98.0,q
1,skypatrol_milliquas:1AXG_J171811_6727,skypatrol_milliquas,1AXG J171811+6727,259.525155,67.450286,QX,0.549,34360723635,,NaN,...,18.22,NaN,NaN,NaN,NaN,NaN,NaN,,98.0,q
2,skypatrol_milliquas:1ES_0502_675,skypatrol_milliquas,1ES 0502+675,76.983942,67.623417,BRX,0.314,34360751539,,NaN,...,17.24,NaN,NaN,NaN,NaN,NaN,NaN,,100.0,q
3,skypatrol_milliquas:1ES_0806_524,skypatrol_milliquas,1ES 0806+524,122.454944,52.316180,BRX,0.138,77309956220,,NaN,...,14.18,NaN,NaN,NaN,NaN,NaN,NaN,,100.0,q
4,skypatrol_milliquas:1ES_1028_511,skypatrol_milliquas,1ES 1028+511,157.827188,50.893284,BRX,0.361,77310874031,,NaN,...,16.23,NaN,NaN,NaN,NaN,NaN,NaN,,100.0,q


MilliQuas feature status counts:


,nuc_feature_status,n
0,ok,24
1,missing_lc_path,1


MilliQuas primary band used:


,primary_band_used,n
0,g,25


,nuc_rank,source_name,primary_band_used,nuc_outlier_label,nuc_outlier_score,nuc_n_points,nuc_median_mag,nuc_feature_status
0,1,1WGA J0943.6+4823,g,transient_like_excursion,21.711838,84.0,18.248671,ok
1,2,1WGA J0909.5+5030,g,transient_like_excursion,17.593646,226.0,17.992505,ok
2,3,1WGA J1052.6+5724,g,transient_like_excursion,12.662702,274.0,17.790284,ok
3,4,1WGA J0952.3+0803,g,transient_like_excursion,12.485998,624.0,17.665311,ok
4,5,1WGA J0905.0+3413,g,transient_like_excursion,12.338414,183.0,17.806037,ok
5,6,1RXS J141901.8+773229,g,transient_like_excursion,12.262818,109.0,17.973204,ok
6,7,1WGA J1047.2+5419,g,transient_like_excursion,11.714764,104.0,18.484368,ok
7,8,1AXG J171811+6727,g,transient_like_excursion,11.570118,100.0,17.874388,ok
8,9,1RXS J14545+0802,g,transient_like_excursion,11.412771,1002.0,16.598048,ok
9,10,1ES 1659+341,g,transient_like_excursion,11.035799,145.0,17.959846,ok


Wrote/resumed MilliQuas outputs under: /Users/calder/code/malca/output/runs/agn_stochastic_milliquas
